> **BLDC copy (voltage-scaled comparison).** This is `dana-mpc-copy-with-estimator/PlantModels.ipynb` with the robot's own motor, the 5010 360 KV BLDC as a DC-equivalent motor (`robot_constants.py`), its driver current limits, and the controller and test voltages rescaled for it. Everything else, the COG included, is unchanged, so the results compare with the Pololu copy. Every change is marked `BLDC:` and listed in `README.md` in this folder.

# This document contains a few robot PLANT simulation **classes** and verifies their behaviour is a simple **open loop** simulation
* Each Robot Class implements a modular physics engine of the plant that can be executed at each time step $\Delta T_{model}$, and contains its own State Vector class. 
* Each Robot Class contains `step`, `get_Hmatrix` and `_init_` Functions that passes the same variables back for drop in place substitution.  
* All numerical paramaters are not defined inside classes and furthermore all classes use the same paramater dictionary `Simulation_Paramaters`.
* All robot classes use 4 inputs (1 per wheel)

1. `OmniRobotSimulationExplicit` used `OmniRobotStateExplicit`:
    * Preserves and moves integrated memory states: armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, and position coordinates
    * Evaluates LuGre bristle deflection states $\mathbf{z}_{k+1}$ using an **unconditionally stable exponential discrete update**, computing continuous traction forces and smoothed rolling drag.
    * Integrates Newton-Euler rigid chassis motion and maps body velocities into global coordinates.
2. `OmniRobotSimulationODESolve` uses `OmniRobotStateODESolve`:
    * 
    * 




```text
==========================================================================
                     SPATIAL REFERENCE FRAME LOGIC
==========================================================================

   GLOBAL FRAME (Fixed Room)               LOCAL FRAME (Robot Centric)
   
          ^+Y_global                              ^+Y_robot (Forward)
          |                                       |
          |                                       |
          +-------> +X_global                     +-------> +X_robot (Right/Strafing)
         /                                       /
        v +Z_global (Up)                        v +Z_robot (Up)

   • Orientation Heading (θ): Angle measured counter-clockwise from +Y_global to +Y_robot.
   • Wheel 1: Front Right (Quadrant I)   at angle +alpha_f relative to +X_robot
   • Wheel 2: Front Left  (Quadrant II)  at angle +alpha_f relative to -X_robot
   • Wheel 3: Rear Left   (Quadrant III) at angle +alpha_b relative to -X_robot
   • Wheel 4: Rear Right  (Quadrant IV)  at angle +alpha_b relative to +X_robot
==========================================================================
```


In [ ]:
class OmniRobotObservableState:
    """
    Data container of data that the controller observes at every time step.
    """
    def __init__(self):
        # Memory Integration States (Differential Cores)
        self.I = np.zeros(4)            # Armature currents [i1, i2, i3, i4] (A)
        self.omega = np.zeros(4)        # Wheel angular velocities [w1, w2, w3, w4] (rad/s)
        self.pos_global = np.zeros(3)   # Global coordinates [x, y, theta] (m, rad)
        self.V_actual = np.zeros(4)     # Clipped applied terminal voltages [V1, V2, V3, V4] (V)
        


### `OmniRobotSimulationExplicit`  Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **hardware-calibrated motor parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit exponential discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters
* $M_r$: Total mass of the robot chassis (kg)
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis (kg·m²)
* $R_b$: Geometric base radius from Center of Mass (CoM) to wheel contact points (m)
* $r$: Operating radius of omni-wheel tires (m)
* $\alpha_f$: Front wheel mounting angle relative to local $+x_r$ axis (rad)
* $\alpha_b$: Rear wheel mounting angle relative to local $+x_r$ axis (rad)
* $h$: Height of Center of Mass (CoM) above ground plane (m)

#### ⚡ Motor Specifications & Derived Parameters
* **Motor Drive**: Brushed DC Motor with integrated gear reduction and H-bridge drive electronics.
* $R$: Armature winding resistance ($\Omega$)
* $L_m$: Armature winding inductance (H)
* $K_t, K_b$: Electromechanical torque constant (N·m/A) and back-EMF constant (V/(rad/s))
* $J_w$: Reflected rotational inertia of motor rotor, gearhead, and wheel assembly (kg·m²)
* $V_{\max}$: Terminal supply voltage saturation limit (V)
* $I_{\max}$: Motor driver over-current protection ceiling (A)
* $C_{rr}$: Floor-tire rolling resistance coefficient
* $\mu_s, \mu_k$: Static and kinetic floor-tire friction coefficients
* $\delta$: Presliding bristle deflection limit (m). **This explicit plant is created with 100 µm**: every place that creates it passes `dict(simulation_parameters, lugre_deflection=100e-6)`. With 2 µm the bristle-wheel loop is too stiff for the explicit step and the wheel speeds reverse on most steps. The ODESolve plant (backward Euler) uses `lugre_deflection` as set (2 µm).
* $v_c$: Creep regularization velocity preventing division by zero at rest (m/s)
* $\alpha_d$: Stribeck exponential decay rate from static to kinetic friction (s/m)
* $\varepsilon$: Hyperbolic tangent smoothing speed for rolling resistance drag
* $v_{s,\text{limit}}$: Wheel sliding velocity threshold limit (delineates pure rolling / presliding from sliding) (rad/s)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts).

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: LuGre normalized bristle deflections.
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw).
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s).
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (m, m, rad).

---

### 3. Kinematic Mapping Matrix $\mathbf{H}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

$$\mathbf{v}_{w,k} = \mathbf{H} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H} = \begin{bmatrix} -\sin\alpha_f & \cos\alpha_f & R_b \\ -\sin\alpha_f & -\cos\alpha_f & R_b \\ \sin\alpha_b & -\cos\alpha_b & R_b \\ \sin\alpha_b & \cos\alpha_b & R_b \end{bmatrix}$$

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f, \qquad y_b = R_b \sin\alpha_b, \qquad L = y_f + y_b$$
$$x_f = R_b \cos\alpha_f, \qquad x_b = R_b \cos\alpha_b, \qquad \Sigma X^2 = x_f^2 + x_b^2$$

#### Static Normal Force Distribution at Rest
$$F_{n,\text{front, static}} = \frac{M_r g y_b}{2 L}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_f}{2 L}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_b}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_f}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} + w_{F,1} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} + w_{F,2} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} + w_{F,3} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} + w_{F,4} \end{bmatrix}$$

---

### 5. LuGre Bristle Friction & Discrete Integration

1. **Contact Slip Velocity along Rolling Axis:**
   $$v_{s,i,k} = (\mathbf{H} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k}$$

2. **Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

3. **Stribeck Friction Curve & Local Characteristic Time Constant:**
   $$\mu_{d,i,k} = \mu_k + (\mu_s - \mu_k) e^{-\alpha_d |v_{s,i,k}|}$$
   $$z_{\text{ss},i,k} = \frac{v_{s,i,k} \mu_{d,i,k}}{\mu_s \sqrt{v_{s,i,k}^2 + v_c^2}}, \qquad \tau_{i,k} = \frac{\delta \mu_{d,i,k}}{\mu_s \sqrt{v_{s,i,k}^2 + v_c^2}}$$

4. **Implicit Exponential Discrete Bristle Update ($\Delta T = 1\text{ ms}$):**
   $$z_{i,k+1} = z_{i,k} e^{-\Delta T / \tau_{i,k}} + z_{\text{ss},i,k} \left( 1 - e^{-\Delta T / \tau_{i,k}} \right)$$

5. **Net Tractive Floor Force & Unified Wheel Speed Update:**
   $$F_{i,k} = -\mu_s F_{n,i,k} z_{i,k+1}$$
   The full LuGre model's bristle damping ($\sigma_1$) and viscous friction ($\sigma_2$) are removed: modified Dahl, the same friction law as the ODESolve plant.
   $$\omega_{i,k+1} = \omega_{i,k} + \frac{\Delta T}{J_w} \left( T_{m,i,k} - T_{\text{roll},i,k} - r F_{i,k} \right)$$

6. **Rigid Chassis, global frame:** the wheel forces act along body directions; Newton's law holds in the global frame, so the global velocity is advanced and the local one is derived from it (changed in this copy: it advanced the local velocity, which leaves out the turning of the robot's axes):
   $$\mathbf{\ddot{X}}_{g,k} = \mathbf{R}(\theta_k)\,\mathbf{M}^{-1}\mathbf{H}^\top \mathbf{F}_k, \qquad \mathbf{\dot{X}}_{g,k+1} = \mathbf{\dot{X}}_{g,k} + \Delta T\,\mathbf{\ddot{X}}_{g,k}, \qquad \mathbf{X}_{g,k+1} = \mathbf{X}_{g,k} + \Delta T\,\mathbf{\dot{X}}_{g,k+1}, \qquad \mathbf{\dot{X}}_{r,k+1} = \mathbf{R}(\theta_{k+1})^\top \mathbf{\dot{X}}_{g,k+1}$$

</details>


---
<details>
<summary><b> Open this for simulation flowchart </b></summary>


```text
==================================================
        SIMULATION STATE DATA (STEP K)
==================================================
 [Inputs] -> Voltages Vector V_1..4
 [Memory] -> Current i_k, Wheel Speeds omega_k
 [Memory] -> LuGre Bristle Deflections z_k
 [Memory] -> Global Chassis Speeds & Past Accel (a_x, a_y)
 [Memory] -> Global Coordinates [x, y, theta]
==================================================
                        |
                        v
  +--------------------------------------------+
  | Step 5.1: Motor Armature Current Update    |
  | di/dt = (V - R*i - Kb*omega) / Lm          |
  | i_{k+1} = i_k + dt * di/dt                 |
  | T_m = Kt * i_{k+1}                         |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.2: Dynamic Normal Force Distribution|
  | (x Right, y Forward Frame)                 |
  | Static load + Pitch (ay) + Roll (ax) + w_F |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.3: LuGre Bristle Friction Update    |
  | v_s = H*v_local - r*omega                  |
  | Exponential Discrete Update z_{k+1}        |
  | F_floor = -mu_s*Fn*z_{k+1}                 |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.4 & 5.5: Rigid Chassis Kinetics     |
  | a_local = M_inv * H^T * F_floor            |
  | a_global = R(theta_k) * a_local            |
  | v_global_{k+1} = v_global_k + dt*a_global  |
  | pos_{k+1} = pos_k + dt * v_global_{k+1}    |
  | v_local_{k+1} = R(theta_{k+1})^T v_global  |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.6: Wheel Speed Integration          |
  | d_omega/dt = (T_m - T_roll - r*F_floor)/Jw |
  | omega_{k+1} = omega_k + dt * d_omega/dt    |
  +--------------------------------------------+
                        |
                        v
==================================================
        ADVANCE STATE TO STEP K+1
==================================================
```
</details>


#### `OmniRobotSimulationExplicit` Class Architecture


*   **`OmniRobotStateExplicit`**: Data container that preserves and moves integrated memory states (armature currents, wheel speeds, LuGre bristle deflections, local/global velocities, and position coordinates) from step $k$ to $k+1$.
*   **`__init__`**: Sets hardware constants, calculates lever arms ($x_f, x_b, y_f, y_b$), builds the $4 \times 3$ kinematic transformation matrix $\mathbf{H}$, and builds the inverse rigid-body mass matrix $\mathbf{M}^{-1}$.
*   **`_build_kinematics_matrix`**: Constructs the $4 \times 3$ matrix projecting 3-DOF body velocity into individual wheel rolling velocities.
*   **`compute_electrical_subsystem`**: Integrates motor currents $\mathbf{I}_{k+1}$ via Forward Euler and calculates gross motor torques $\mathbf{T}_{m,k}$.
*   **`compute_dynamic_normal_forces`**: Evaluates vertical normal loads $\mathbf{F}_{n,k}$ with pitch/roll load shifts based on past accelerations ($a_{x,\text{past}}, a_{y,\text{past}}$) and vibration noise.
*   **`compute_contact_friction_interface`**: Evaluates LuGre bristle deflection states $\mathbf{z}_{k+1}$ using an **unconditionally stable exponential discrete update**, computing continuous traction forces and smoothed rolling drag.
*   **`advance_chassis_kinetics`**: Integrates Newton-Euler rigid chassis motion in the global frame: the wheel forces are rotated into the global frame, the global velocity is advanced, and the local velocity is derived from it (changed in this copy).
*   **`step`**: Executes the master pipeline loop for one discrete time step $\Delta T$.

In [ ]:
import numpy as np

class OmniRobotStateExplicit:
    """
    Data container to log and preserve the electromechanical system states 
    at any discrete time step k.
    """
    def __init__(self):
        # Memory Integration States (Differential Cores)
        self.I = np.zeros(4)            # Armature currents [i1, i2, i3, i4] (A)
        self.omega = np.zeros(4)        # Wheel angular velocities [w1, w2, w3, w4] (rad/s)
        self.z = np.zeros(4)            # LuGre normalized bristle deflections [z1, z2, z3, z4]
        self.v_local = np.zeros(3)      # Local chassis velocities [vx (Right), vy (Forward), omega] (m/s, rad/s)
        self.a_local_past = np.zeros(3) # Historical chassis accelerations from step k-1 (m/s^2, rad/s^2)
        self.v_global = np.zeros(3)     # Global velocities [vx_g, vy_g, omega_g] (m/s, rad/s)
        self.pos_global = np.zeros(3)   # Global coordinates [x, y, theta] (m, rad)
        self.V_actual = np.zeros(4)     # Clipped applied terminal voltages [V1, V2, V3, V4] (V)
        
        # Tracked Logical Status Vector 
        self.S = np.zeros(4, dtype=bool) # Slip threshold indicators (False=Below slip band, True=Exceeds slip band)


class OmniRobotSimulationExplicit:
    """
    Physics engine plant managing an asymmetric 4-Omni wheel robot model
    with dynamic weight transfer and LuGre bristle floor friction boundaries.
    """
    def __init__(self, params: dict, random_seed: int = 42):
        """
        Initializes hardware constants, geometric parameters, and noise variances.
        """
        self.p = params
        self.dt = params['dt']
        
        # Pre-compute fixed kinematic tracking matrix (H)
        self.H = self._build_kinematics_matrix()

        # Dedicated random number generator state
        self.rng = np.random.default_rng(seed=random_seed)

        # --- BUILD THE MASS / INERTIA MATRIX (M) ---
        self.M = np.array([
            [self.p['Mr'], 0.0,           0.0],          # Mass acting on X acceleration (Right)
            [0.0,          self.p['Mr'],  0.0],          # Mass acting on Y acceleration (Forward)
            [0.0,          0.0,           self.p['Jzz']] # Rotational inertia acting on Alpha
        ])
        
        self.M_inv = np.linalg.inv(self.M)
        
        # Structural wheelbase lengths for weight distribution balances (+x Right, +y Forward)
        self.y_f = self.p['Rb'] * np.sin(self.p['alpha_f'])
        self.y_b = self.p['Rb'] * np.sin(self.p['alpha_b'])
        self.x_f = self.p['Rb'] * np.cos(self.p['alpha_f'])
        self.x_b = self.p['Rb'] * np.cos(self.p['alpha_b'])
        self.L = self.y_f + self.y_b
        self.sum_X2 = (self.x_f**2) + (self.x_b**2)

    def _build_kinematics_matrix(self) -> np.ndarray:
        """Constructs the exact asymmetric velocity projection transformation matrix H."""
        af = self.p['alpha_f']
        ab = self.p['alpha_b']
        Rb = self.p['Rb']
        return np.array([
            [-np.sin(af),  np.cos(af), Rb],  # Wheel 1: Front Right
            [-np.sin(af), -np.cos(af), Rb],  # Wheel 2: Front Left
            [ np.sin(ab), -np.cos(ab), Rb],  # Wheel 3: Rear Left
            [ np.sin(ab),  np.cos(ab), Rb]   # Wheel 4: Rear Right
        ])

    def compute_electrical_subsystem(self, state: OmniRobotStateExplicit, V_input: np.ndarray) -> np.ndarray:
        """
        Calculates Step 5.1: Enforces Voltage Hardpoint Saturation Bounds (V_max) 
        and Over-Current Protection Hardpoint Bounds (I_max). Advances motor armature 
        currents using Forward Euler discrete integration and maps them to gross 
        electromagnetic torques.
        """
        Lm = self.p['Lm']
        R = self.p['R']
        Kb = self.p['Kb']
        Kt = self.p['Kt']
        V_max = self.p.get('V_max', 12.0)
        I_max = self.p.get('I_max', 20.0)
        
        # 1. Voltage Hardpoint Clipping (Power Supply / H-Bridge Saturation Limit)
        V_clipped = np.clip(V_input, -V_max, V_max)
        state.V_actual = V_clipped
        
        # 2. Back-EMF & Current Derivative Calculation
        v_b = Kb * state.omega
        di_dt = (V_clipped - R * state.I - v_b) / Lm
        
        # 3. Discrete Current Integration with Over-Current Hardpoint Clipping
        I_unclipped = state.I + self.dt * di_dt
        state.I_next = np.clip(I_unclipped, -I_max, I_max)
        
        # 4. Gross Electromagnetic Motor Torque Output
        T_m = Kt * state.I_next
        return T_m

    def compute_dynamic_normal_forces(self, state: OmniRobotStateExplicit) -> np.ndarray:
        """
        Calculates Step 5.2: Determines the vertical load profile for each wheel 
        by balancing static weight distributions with dynamic multi-axis pitch/roll 
        inertial moments (+x Right, +y Forward), plus random structural vibration noise.
        """
        Mr = self.p['Mr']
        g = self.p['g']
        h = self.p['h']
        L = self.L
        x_f = self.x_f
        x_b = self.x_b
        y_f = self.y_f
        y_b = self.y_b
        sum_X2 = self.sum_X2
        sigma_V = self.p['sigma_V']
        
        ax_past = state.a_local_past[0]  # Right
        ay_past = state.a_local_past[1]  # Forward
        
        # Base Static Distributions
        F_n_front_static = (Mr * g * y_b) / (2.0 * L)
        F_n_rear_static  = (Mr * g * y_f) / (2.0 * L)
        
        # Dynamic Weight Shifts
        delta_Fn_y = (Mr * ay_past * h) / (2.0 * L)  # Pitch due to forward accel
        lateral_moment_scale = (Mr * ax_past * h) / L
        delta_Fn_x_front = (x_f / sum_X2) * lateral_moment_scale * y_b  # Roll
        delta_Fn_x_rear  = (x_b / sum_X2) * lateral_moment_scale * y_f
        
        w_F = self.rng.normal(0.0, sigma_V, 4) if sigma_V > 0 else np.zeros(4)
        
        F_n = np.zeros(4)
        F_n[0] = F_n_front_static - delta_Fn_y - delta_Fn_x_front + w_F[0] # Wheel 1: Front Right
        F_n[1] = F_n_front_static - delta_Fn_y + delta_Fn_x_front + w_F[1] # Wheel 2: Front Left
        F_n[2] = F_n_rear_static  + delta_Fn_y + delta_Fn_x_rear  + w_F[2] # Wheel 3: Rear Left
        F_n[3] = F_n_rear_static  + delta_Fn_y - delta_Fn_x_rear  + w_F[3] # Wheel 4: Rear Right
        
        return np.maximum(0.0, F_n)

    def compute_contact_friction_interface(self, T_m: np.ndarray, F_n: np.ndarray, state: OmniRobotStateExplicit) -> tuple:
        """
        Calculates Step 5.3: Evaluates LuGre bristle floor patch friction mechanics.
        Uses exact exponential discrete steps for stiff bristle deflection state.
        """
        r = self.p['r']
        Crr = self.p['Crr']
        mu_s_nom = self.p['mu_s']
        mu_k_nom = self.p['mu_k']
        sigma_mu = self.p['sigma_mu']
        delta_l = self.p['lugre_deflection']
        v_c = self.p['lugre_creep']
        alpha_d = self.p['alpha_d']
        roll_smoothing = self.p['roll_smoothing']
        
        dt = self.dt
        
        w_mu = self.rng.normal(0.0, sigma_mu, 4) if sigma_mu > 0 else np.zeros(4)
        mu_s = np.maximum(0.0, mu_s_nom + w_mu)
        mu_k = np.maximum(0.0, mu_k_nom + w_mu)
        
        v_contact = self.H @ state.v_local
        v_s = v_contact - r * state.omega
        
        T_roll = Crr * F_n * r * np.tanh(state.omega / roll_smoothing)
        
        mu_d = mu_k + (mu_s - mu_k) * np.exp(-alpha_d * np.abs(v_s))
        denom = np.sqrt(v_s**2 + v_c**2)
        z_ss = (v_s * mu_d) / (mu_s * denom)
        tau = (delta_l * mu_d) / (mu_s * denom)
        
        exp_factor = np.exp(-dt / tau)
        z_next = state.z * exp_factor + z_ss * (1.0 - exp_factor)
        
        # Bristle damping (sigma_1) and viscous friction (sigma_2) removed: modified Dahl, as in the ODESolve plant
        F_floor = -mu_s * F_n * z_next
        
        T_load = -F_floor * r + T_roll
        
        return F_floor, T_load, z_next, T_roll

    def advance_chassis_kinetics(self, F_floor: np.ndarray, state: OmniRobotStateExplicit) -> OmniRobotStateExplicit:
        """
        Calculates Step 5.4 & 5.5: Advances rigid chassis velocities and global 
        position using Newton-Euler equations.
        """
        dt = self.dt
        theta_k = state.pos_global[2]
        R_theta = OmniRobotSimulationODESolve._rotation(theta_k)        # GLOBAL FRAME: the plant's rotation

        # GLOBAL FRAME: the wheel forces act along body directions, so the acceleration starts in the local frame.
        # Newton's law holds in the global frame, so it is rotated there and the GLOBAL velocity is advanced.
        # (Advancing the local velocity leaves out the turning of the robot's axes.)
        a_local = self.M_inv @ (self.H.T @ F_floor)
        a_global = R_theta @ a_local
        v_global_next = state.v_global + dt * a_global
        pos_global_next = state.pos_global + dt * v_global_next

        next_state = OmniRobotStateExplicit()
        next_state.v_global = v_global_next
        next_state.pos_global = pos_global_next
        next_state.v_local = OmniRobotSimulationODESolve._rotation(pos_global_next[2]).T @ v_global_next   # GLOBAL FRAME: local derived, new heading
        next_state.a_local_past = a_local

        return next_state

    def step(self, current_state: OmniRobotStateExplicit, V_input: np.ndarray) -> tuple:
        """
        Executes the entire sequential discrete physics pipeline for a single time step dt.
        """
        r = self.p['r']
        Jw = self.p['Jw']
        dt = self.dt
        
        T_m = self.compute_electrical_subsystem(current_state, V_input)
        F_n = self.compute_dynamic_normal_forces(current_state)
        F_floor, T_load, z_next, T_roll = self.compute_contact_friction_interface(T_m, F_n, current_state)
        
        next_state = self.advance_chassis_kinetics(F_floor, current_state)
        next_state.I = current_state.I_next
        next_state.V_actual = current_state.V_actual
        next_state.z = z_next
        
        d_omega_dt = (T_m - T_roll - r * F_floor) / Jw
        next_state.omega = current_state.omega + dt * d_omega_dt
        
        return next_state
    
    def get_observable_state( self, full_state: OmniRobotStateExplicit ) -> OmniRobotObservableState: 
        obs = OmniRobotObservableState() 
        obs.pos_global = full_state.pos_global.copy() 
        obs.omega = full_state.omega.copy() 
        obs.I = full_state.I.copy() 
        obs.V_actual = full_state.V_actual.copy()
        return obs


## `OmniRobotSimulationODESolve` Plant Simulation Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **non-calibrated hardware parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit (backward Euler) discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters (Henry Calibrated Constants)
* $M_r$ : Total mass of the robot
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis
* $J_w $: Rotational moment of inertia of a wheel/rotor assembly
* $R_b$ : Geometric base radius from the base centre to the wheel contact points
* $r $: Operating radius of omni-wheel tires
* $\alpha_f$ = : Front wheel mounting angle relative to local $+x_r$ axis
* $\alpha_b $= : Rear wheel mounting angle relative to local $+x_r$ axis
* $h$: Height of Center of Mass (CoM) above ground plane
* $y_{cg}$ : Center of Mass position forward of the base centre (0 for now: centred)
* $V_{max}$ : Supply voltage saturation limit
* $R$ , $L_m$ : DC motor armature winding resistance and inductance
* $K_t $= $K_b$ : Electromechanical torque and back-EMF constants
* $C_{rr}$ : Floor-tire rolling resistance coefficient
* $\mu_s$ , $\mu_k$ : Static and kinetic floor-tire friction coefficients
* $\delta$ : Presliding bristle deflection limit ($2\,\mu\text{m}$)
* $v_c $: Creep regularization velocity preventing division by zero at rest
* $\alpha_d $: Stribeck exponential decay rate from static to kinetic friction
* $\varepsilon $: Hyperbolic tangent smoothing speed for rolling resistance drag
* $\mu_{a,s}$, $\mu_{a,k}$, $\alpha_a$, $k_a$ : Roller (a-axis) friction coefficients at zero and at high axial slip, their decay rate, and the slope of the smoothed sign (change 9)
* $\epsilon_{tol}$, $n_{max}$ : Tolerance on the slip change, and the most solves per step, for the wheel solve (`implicit_tol`, `implicit_max_iter`)
* $\sigma_V$, $\sigma_\mu$ : Standard deviations of the wheel load noise $w_F$ and the floor friction noise $w_\mu$ (`sigma_V`, `sigma_mu`)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts), clipped to $\pm V_{max}$.

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: Normalized bristle deflections (meters / normalized).
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw), derived from the global velocities.
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s): the advanced velocity state, since Newton's law holds in this fixed frame.
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (meters, meters, radians).

---

### 3. Kinematic Mapping Matrices $\mathbf{H}_{drive}$, $\mathbf{H}_{axial}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

The chassis motion is measured about the Center of Mass, $y_{cg}$ forward of the base centre; with $\mathbf{p}_i$ each wheel's contact point from it. Each row of a matrix is a wheel's direction followed by its yaw lever: $\mathbf{H}_{drive}$ has rows $[\mathbf{d}_i^\top,\ (\mathbf{p}_i \times \mathbf{d}_i)_z]$ and $\mathbf{H}_{axial}$ has rows $[\mathbf{a}_i^\top,\ (\mathbf{p}_i \times \mathbf{a}_i)_z]$:

The drive direction is the axial direction turned $90^\circ$:

$$\delta_f = \alpha_f + 90^\circ, \qquad \delta_b = \alpha_b + 90^\circ$$

$$\mathbf{v}_{w,k} = \mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{drive} = \begin{bmatrix} \cos\delta_f & \sin\delta_f & R_b + y_{cg}\cos\delta_f \\ \cos\delta_f & -\sin\delta_f & R_b + y_{cg}\cos\delta_f \\ -\cos\delta_b & -\sin\delta_b & R_b - y_{cg}\cos\delta_b \\ -\cos\delta_b & \sin\delta_b & R_b - y_{cg}\cos\delta_b \end{bmatrix}$$

$$\mathbf{v}_{a,k} = \mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{axial} = \begin{bmatrix} \cos\alpha_f & \sin\alpha_f & y_{cg}\cos\alpha_f \\ -\cos\alpha_f & \sin\alpha_f & -y_{cg}\cos\alpha_f \\ -\cos\alpha_b & -\sin\alpha_b & -y_{cg}\cos\alpha_b \\ \cos\alpha_b & -\sin\alpha_b & y_{cg}\cos\alpha_b \end{bmatrix}$$

$J_{zz}$ is the yaw inertia about the Center of Mass, so the mass matrix is $\mathbf{M} = \text{diag}(M_r, M_r, J_{zz})$.

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f = 0.03975\text{ m}, \qquad y_b = R_b \sin\alpha_b = 0.05622\text{ m}, \qquad L = y_f + y_b = 0.09597\text{ m}$$
$$x_f = R_b \cos\alpha_f = 0.06885\text{ m}, \qquad x_b = R_b \cos\alpha_b = 0.05622\text{ m}, \qquad \Sigma X^2 = x_f^2 + x_b^2 = 0.007901\text{ m}^2$$
The front and rear wheel distances from the Center of Mass (equal to $y_f$, $y_b$ while it is centred):
$$y_{f,cg} = y_f - y_{cg}, \qquad y_{b,cg} = y_b + y_{cg}$$

#### Static Normal Force Distribution at Rest (values with the CoG centred)
$$F_{n,\text{front, static}} = \frac{M_r g y_{b,cg}}{2 L} = 7.752\text{ N}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_{f,cg}}{2 L} = 5.482\text{ N}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{b,cg}}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{f,cg}}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} \end{bmatrix}$$

The robot adds the load disturbance $w_{F,i,k} \sim \mathcal{N}(0, \sigma_V^2)$, drawn each step inside `compute_dynamic_normal_forces`, and a wheel cannot pull on the floor:
$$F_{n,i,k} \leftarrow \max\left(0,\ F_{n,i,k} + w_{F,i,k}\right)$$

---

### 5. Motor, Bristle Friction (Modified Dahl) & Discrete Integration

The motor current, wheel speed and bristle deflection of each wheel change faster than $\Delta T$ (the bristles' presliding and the motor's $L_m/R$), so they are discretised with **backward (implicit) Euler**, $\frac{dx}{dt} \approx \frac{x_{k+1} - x_k}{\Delta T}$ with the right-hand side at step $k+1$, which stays stable at any $\Delta T$. The three are solved together for each wheel (item 7). The chassis then follows with Forward Euler.

**Step 5.1: Supply Limit and Motor Armature Current:**
   $$V_{i,k} = \text{clip}\left(V_{cmd,i,k}, -V_{max}, V_{max}\right), \qquad L_m \frac{i_{i,k+1} - i_{i,k}}{\Delta T} = V_{i,k} - R\, i_{i,k+1} - K_b\, \omega_{i,k+1}, \qquad T_{m,i,k+1} = K_t\, i_{i,k+1}$$

**Step 5.2: Contact Slip Velocity along Rolling Axis** (chassis contact speed of step $k$, new wheel speed):
   $$v_{s,i,k+1} = (\mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k+1}$$

**Step 5.3: Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

**Step 5.4: Stribeck Friction Curve:**
   The friction coefficients include the floor friction noise $w_{\mu,i,k} \sim \mathcal{N}(0, \sigma_\mu^2)$, drawn once per step and held for every pass of the wheel solve (Step 5.7):
   $$\mu_{s,i} = \max\left(0,\ \mu_s + w_{\mu,i,k}\right), \qquad \mu_{k,i} = \max\left(0,\ \mu_k + w_{\mu,i,k}\right)$$
   $$\mu_{d,i} = \mu_{k,i} + (\mu_{s,i} - \mu_{k,i}) e^{-\alpha_d |v_{s,i,k+1}|}$$
   In Steps 5.5 to 5.7, $\mu_s$ is each wheel's $\mu_{s,i}$.

**Step 5.5: Backward Euler Bristle Update ($\Delta T = 1\text{ ms}$):**
   The LuGre bristle equation:
   $$\delta \, \dot{z}_i = v_{s,i} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}\; z_i$$
   Over one step, with backward Euler:
   $$\delta \, \frac{z_{i,k+1} - z_{i,k}}{\Delta T} = v_{s,i,k+1} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i,k+1}^2 + v_c^2}\; z_{i,k+1}$$

**Step 5.6: Net Tractive Floor Force:**
   $$F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$$
   The full LuGre model has other terms (bristle damping and viscous friction). They are removed here: this is a simple model, and all it needs is the bristle deflection term, so that the force is not zero at zero slip. This makes it a modified Dahl model: LuGre with $\sigma_1 = \sigma_2 = 0$, as in Dahl, but keeping the Stribeck curve, where strict Dahl sets $\mu_s = \mu_d$ ([Altair: LuGre and Dahl friction](https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm)).
   In sliding the bristles settle within a step, $F_i = -\text{sign}(v_{s,i})\, \mu_{d,i} F_{n,i}$: the Coulomb–Stribeck law.

**Step 5.7: Unified Wheel Speed Update (solved with Steps 5.1, 5.5 and 5.6):**
   $$J_w \frac{\omega_{i,k+1} - \omega_{i,k}}{\Delta T} = T_{m,i,k+1} - T_{\text{roll},i,k} - r F_{i,k+1}$$
   With $v_{s,i,k+1} = v_{w,i,k} - r\,\omega_{i,k+1}$ (Step 5.2), $F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$ (Step 5.6) and $d_i = \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}$ held fixed, the motor (Step 5.1), wheel (this step) and bristle (Step 5.5) equations are linear in the three new values. Moving those to the left gives one matrix equation per wheel, $\mathbf{A}_i \mathbf{x}_{i,k+1} = \mathbf{b}_i$, with one row per equation:
   $$\underbrace{\begin{bmatrix} L_m/\Delta T + R & K_b & 0 \\ -K_t & J_w/\Delta T & -r \mu_s F_{n,i,k} \\ 0 & r & \delta/\Delta T + d_i \end{bmatrix}}_{\mathbf{A}_i} \underbrace{\begin{bmatrix} i_{i,k+1} \\ \omega_{i,k+1} \\ z_{i,k+1} \end{bmatrix}}_{\mathbf{x}_{i,k+1}} = \underbrace{\begin{bmatrix} (L_m/\Delta T)\, i_{i,k} + V_{i,k} \\ (J_w/\Delta T)\, \omega_{i,k} - T_{\text{roll},i,k} \\ (\delta/\Delta T)\, z_{i,k} + v_{w,i,k} \end{bmatrix}}_{\mathbf{b}_i} \quad \begin{matrix} \text{motor} \\ \text{wheel} \\ \text{bristle} \end{matrix}$$
   It is solved by elimination, for all four wheels at once: the motor row gives $i_{i,k+1}$ and the bristle row gives $z_{i,k+1}$ in terms of $\omega_{i,k+1}$, and putting both into the wheel row leaves one equation for $\omega_{i,k+1}$. This is the same answer as solving $\mathbf{A}_i \mathbf{x}_{i,k+1} = \mathbf{b}_i$, without a matrix solve per wheel.
   The chassis is held during the solve, so the wheels do not affect each other. $d_i$ depends on the new slip through $\mu_{d,i}$, so the equation is solved again with $d_i$ updated at the new slip until the slip changes by less than $\epsilon_{tol}$ (at most $n_{max}$ solves). The load torque on each motor is $T_{load,i,k} = r F_{i,k+1} + T_{\text{roll},i,k}$.

**Step 5.8: Roller (a-axis) Friction — explicit, from the chassis velocity of step $k$ (change 9):** each wheel's rollers let it slide along its axial direction $\mathbf{a}_i$ (the drive direction $\mathbf{d}_i$ turned $-90^\circ$), and the roller bushings resist that sliding.:
   $$v_{a,i,k} = (\mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k})_i, \qquad \mu_{a,i} = \mu_{a,k} + (\mu_{a,s} - \mu_{a,k})\, e^{-\alpha_a |v_{a,i,k}|}$$
   $$F_{a,i,k} = -F_{n,i,k}\, \mu_{a,i}\, \frac{2}{\pi} \arctan\left(k_a\, v_{a,i,k}\right), \qquad \mathbf{W}_{a,k} = \mathbf{H}_{axial}^\top \mathbf{F}_{a,k}$$
   The velocity of step $k$ is held over the step: a one-step lag, which is stable because this force is not stiff (it changes the chassis velocity at most about 160 /s, against $1/\Delta T$ = 1000 /s). The roller force lies along the wheel's axial, so it does not turn the wheel; it acts on the chassis only. **Off by default:** `roller_friction` in `simulation_parameters`.

**Step 5.9: Rigid Chassis Kinetics (global room frame):** the wheels are fixed to the chassis, so their forces act along body directions; Newton's law holds in the fixed room frame, so the global velocities are advanced there with the heading $\theta_k$:
   $$\mathbf{F}_{r,k} = \mathbf{H}_{drive}^\top \mathbf{F}_{k+1} + \mathbf{W}_{a,k}, \qquad \mathbf{F}_{g,k} = \mathbf{R}(\theta_k)\, \mathbf{F}_{r,k}, \qquad \mathbf{\ddot{X}}_{g,k} = \mathbf{M}^{-1} \mathbf{F}_{g,k}$$
   $$\mathbf{\dot{X}}_{g,k+1} = \mathbf{\dot{X}}_{g,k} + \Delta T\, \mathbf{\ddot{X}}_{g,k}, \qquad \mathbf{X}_{g,k+1} = \mathbf{X}_{g,k} + \Delta T\, \mathbf{\dot{X}}_{g,k+1}$$
   $$\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad \mathbf{\dot{X}}_{r,k+1} = \mathbf{R}(\theta_{k+1})^\top \mathbf{\dot{X}}_{g,k+1}, \qquad \mathbf{\ddot{X}}_{r,k} = \mathbf{R}(\theta_k)^\top \mathbf{\ddot{X}}_{g,k}$$
   There is no separate rotation term: Newton's law is written in the fixed room frame, and the local velocity and acceleration are only the global ones rotated into the robot's axes. $\mathbf{\ddot{X}}_{r,k}$ is the global acceleration along the robot's axes, used by the load transfer of the next step; it is not the time derivative of $\mathbf{\dot{X}}_r$.

</details>

---
<details>
<summary><b> Open this for simulation flowchart </b></summary>


```text
==================================================
        SIMULATION STATE DATA (STEP K)
==================================================
 [Inputs] -> Voltages Vector V_1..4
 [Memory] -> Current i_k, Wheel Speeds omega_k
 [Memory] -> Bristle Deflections z_k
 [Memory] -> Global Chassis Speeds & Past Accel (a_x, a_y)
 [Memory] -> Global Coordinates [x, y, theta]
==================================================
                        |
                        v
  +--------------------------------------------+
  | Step 5.1: Motor Armature Current Update    |
  | V = clip(V_cmd, +-V_max)                   |
  | Backward Euler: motor row of the matrix    |
  | in Step 5.7, T_m = Kt * i_{k+1}            |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Section 4: Dynamic Normal Force            |
  | (x Right, y Forward Frame)                 |
  | Static load + Pitch (ay) + Roll (ax)       |
  | + load noise w_F, then clip F_n >= 0       |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Steps 5.4-5.6: Modified Dahl Friction      |
  | w_mu once per step -> mu_s,i and mu_k,i    |
  | v_s = H_drive*v_local_k - r*omega_{k+1}    |
  | Backward Euler: bristle row of the matrix  |
  | in Step 5.7, F_floor = -mu_s*F_n*z_{k+1}   |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.7: Wheel Speed Update               |
  | Jw*(omega_{k+1}-omega_k)/dt =              |
  |      T_m - T_roll - r*F_floor              |
  | With 5.1 and 5.5, per wheel: A x = b,      |
  | solved by elimination, all 4 wheels        |
  | x = [i, omega, z]_{k+1}; repeated until    |
  | the slip settles                           |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.8: Roller (a-axis) Friction         |
  | explicit, chassis velocity of step k       |
  | v_a = H_axial*v_local_k                    |
  | F_a = -F_n*mu_a*(2/pi)*atan(k_a*v_a)       |
  | -> wrench W_roller on the chassis only     |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.9: Rigid Chassis Kinetics           |
  | F_local = H_drive^T*F + W_roller           |
  | F_global = R(theta_k)*F_local              |
  | a_global = M_inv*F_global                  |
  | v_global_{k+1} = v_global_k + dt*a_global  |
  | pos_{k+1} = pos_k + dt * v_global_{k+1}    |
  | v_local_{k+1} = R(theta_{k+1})^T v_global  |
  +--------------------------------------------+
                        |
                        v
==================================================
        ADVANCE STATE TO STEP K+1
==================================================
```
</details>

#### `OmniRobotSimulationODESolve` Class Architecture

The `OmniRobotSimulationODESolve` class implements a modular, discrete-time physics engine executed at each time step $\Delta T$. The robot is created once from `simulation_parameters`; `step` calls its methods in order, and the controller only sees `get_observable_state`. Each robot carries its own parameters, geometry and noise, so several can run side by side.

*   **`OmniRobotStateODESolve`**: Data container that preserves and moves the states (armature currents, wheel speeds, bristle deflections, local/global velocities, past accelerations, position coordinates, applied voltages and slip flags) from step $k$ to $k+1$.
*   **`__init__`**: Stores the parameters and the robot's random number generator, and works out everything fixed once: the wheel directions $\mathbf{d}_i$, $\mathbf{a}_i$ and positions $\mathbf{p}_i$ from the Center of Mass, the $4 \times 3$ kinematic matrices $\mathbf{H}_{drive}$ and $\mathbf{H}_{axial}$ (Section 3), the static wheel loads and the pitch/roll gains (Section 4), and the mass matrix $\mathbf{M}$ and its inverse $\mathbf{M}^{-1}$.
*   **`_rotation`**: The rotation $\mathbf{R}(\theta)$ between the local chassis frame and the global room frame.
*   **`compute_dynamic_normal_forces`**: Evaluates vertical normal loads $\mathbf{F}_{n,k}$ with pitch/roll load shifts based on past accelerations ($a_{x,\text{past}}, a_{y,\text{past}}$) and the vibration noise $w_F$ (Section 4).
*   **`compute_roller_friction`**: Roller (a-axis) friction: each wheel's roller force along its axial direction, and their wrench on the chassis (Step 5.8). On by default.
*   **`compute_contact_friction_interface`**: Draws the floor friction noise $w_\mu$, works out the rolling drag, then solves each wheel's motor current, wheel speed and bristle deflection together from the $3 \times 3$ backward Euler matrix equation, solved by elimination for all four wheels at once, repeated until the slip settles (Steps 5.1, 5.3 to 5.7).
*   **`advance_chassis_kinetics`**: Rotates the wheel forces into the global room frame, applies Newton's law with $\mathbf{M}^{-1}$, and advances the global velocities, then the coordinates (Step 5.9).
*   **`step`**: Executes the master pipeline for one discrete time step $\Delta T$: clips the voltages to the supply, then calls the methods above in order. Returns the next state (Dana's interface; the slip flags are kept in the state as `S`).
*   **`get_observable_state`**: What the controller gets (`OmniRobotObservableState`): the pose, wheel speeds, motor currents and applied voltages.

**About the name:** it is not an ODE solver. Each 1 ms step is backward Euler: the motor, wheel and bristle equations at the end of the step are solved as a set of linear equations (by elimination), repeated until the slip settles. The chassis then follows with forward Euler. The name is kept because Dana's runner and controllers look the class up by it.

In [ ]:
"""
The robot
=========

OmniRobotStateODESolve holds the full state of the robot at one step k. OmniRobotSimulationODESolve is the robot: it
is created once from simulation_parameters, works out everything fixed in __init__, and step advances
the full state by one dt. Several robots can run side by side, each with its own parameters and noise.

From outside, the robot is a black box:
    step(state, V_cmd)            advances the full state by one dt
    get_observable_state(state)   what the controller sees: pose, wheel speeds, currents, applied voltages

Inside, step calls:

    supply clip of the voltage commands
        ↓
    compute_dynamic_normal_forces        wheel loads, with the load noise
        ↓
    compute_roller_friction              roller friction, if switched on
        ↓
    compute_contact_friction_interface   friction noise, then motor, wheel and bristles solved together
        ↓
    advance_chassis_kinetics             chassis acceleration, velocity and pose (global frame)
"""

import numpy as np


class OmniRobotStateODESolve:
    """
    Data container that preserves the states of the robot at any discrete time step k.
    """
    def __init__(self):
        self.I = np.zeros(4)              # motor currents [A]
        self.omega = np.zeros(4)          # wheel speeds [rad/s]
        self.z = np.zeros(4)              # bristle deflections, as a fraction of lugre_deflection
        self.v_global = np.zeros(3)       # global velocity [x_dot, y_dot, theta_dot], the advanced state
        self.v_local = np.zeros(3)        # local velocity [x_dot, y_dot, theta_dot], derived from v_global
        self.pos_global = np.zeros(3)     # pose [X, Y, theta] [m, m, rad]
        self.a_local_past = np.zeros(3)   # chassis acceleration of the previous step, local frame [x, y, yaw]
        self.V_actual = np.zeros(4)       # motor voltages after the supply clip [V]

        # Slip flags: True where a wheel's slip is above slip_band (a report only)
        self.S = np.zeros(4, dtype=bool)


class OmniRobotSimulationODESolve:
    """
    The 4-omni-wheel robot: wheel layout, motors, floor friction and the rigid chassis.
    """

    # =======================================================================
    # Setup: everything fixed, worked out once
    # =======================================================================

    def __init__(self, params: dict, random_seed: int = 42):
        """
        params: simulation_parameters
        random_seed: seed of this robot's noise, so runs are repeatable
        """
        self.p = params
        self.dt = params['dt']

        # Dedicated random number generator for this robot's noise
        self.rng = np.random.default_rng(seed=random_seed)

        # -------------------------------------------------------------------
        # Wheel geometry (Section 3)
        # -------------------------------------------------------------------
        alpha_f = params['alpha_f']
        alpha_b = params['alpha_b']
        Rb = params['Rb']
        cog_y = params['cog_y']

        # Wheel angles around the base circle, counter-clockwise from +x (right). The front wheels sit
        # alpha_f above the x axis and the rear wheels alpha_b below it, mirrored left to right.
        alpha_FR = alpha_f                   # 1 front right
        alpha_FL = np.pi - alpha_f           # 2 front left:  180 deg - alpha_f
        alpha_RL = np.pi + alpha_b           # 3 rear left:   180 deg + alpha_b
        alpha_RR = 2.0 * np.pi - alpha_b     # 4 rear right:  360 deg - alpha_b
        alpha = np.array([alpha_FR, alpha_FL, alpha_RL, alpha_RR])
        delta = alpha + np.pi / 2 

        # d_hat: motor drive direction, 90 deg ahead of sliding direction
        self.d_hat = np.column_stack((np.cos(delta), np.sin(delta)))
        # a_hat: siding direction from rollers (out from the centre). 
        self.a_hat = np.column_stack((np.cos(alpha), np.sin(alpha)))

        # Each wheel's position measured from the CoG (r_i), the point the chassis equations are written about
        self.positions = Rb * self.a_hat - np.array([0.0, cog_y])

        # Yaw levers: z component of r_i x d_hat_i and r_i x a_hat_i
        drive_lever = self.cross2d(self.positions, self.d_hat)
        axial_lever = self.cross2d(self.positions, self.a_hat)

        # H_drive: rows are [drive x, drive y, yaw lever], so drive speeds = H_drive [x_dot, y_dot, theta_dot]
        # H_axial: rows are [axial x, axial y, yaw lever], so axial speeds = H_axial [x_dot, y_dot, theta_dot]
        self.H_drive = np.column_stack((self.d_hat, drive_lever))
        self.H = self.H_drive   # Dana's name for it: her runner and tests use plant.H
        self.H_axial = np.column_stack((self.a_hat, axial_lever))

        # -------------------------------------------------------------------
        # Wheel loads: the fixed parts (Section 4)
        # -------------------------------------------------------------------
        Mr = params['Mr']
        g = params['g']
        h = params['h']

        # Distances from the CoG, all positive. The layout is mirrored left to right, so wheel 1 (FR)
        # gives the front pair's distances and wheel 4 (RR) the rear pair's.
        front_x = self.positions[0, 0]      # front wheels: sideways from the CoG
        front_y = self.positions[0, 1]      #               ahead of the CoG
        rear_x = self.positions[3, 0]       # rear wheels:  sideways from the CoG
        rear_y = -self.positions[3, 1]      #               behind the CoG (wheel 4's y is negative)

        wheelbase = front_y + rear_y        # front pair to rear pair

        # Static split: a lever balance about the CoG. Each pair's share of the weight is the other pair's
        # distance over the wheelbase, so the pair closer to the CoG carries more. Each wheel takes half.
        front_share = rear_y / wheelbase
        rear_share = front_y / wheelbase
        self.F_n_front_static = front_share * Mr * g / 2.0 # Full Robot load / 2 wheels
        self.F_n_rear_static = rear_share * Mr * g / 2.0 # Full Robot load / 2 wheels

        # Pitch: accelerating forward tips the robot back. The tipping moment is Mr * h * a_y, and the front 
        # and rear pairs, one wheelbase apart, push back against it. 
        # Each wheel's load changes by pitch_gain * a_y: the front wheels lose it, the rear gain it.
        self.pitch_gain = (Mr * h) / (2.0 * wheelbase) # Full robot moment / 2 wheel base distances

        # Roll: accelerating right tips the robot to the left. The tipping moment is Mr * h * a_x (mass times
        # CoG height times sideways acceleration), and the left and right wheels push back against it. Each
        # pair takes the same share of it as of the weight, spread over the pairs in proportion to their
        # sideways distance. Each wheel's load changes by roll_gain * a_x: the right wheels lose it, the left
        # gain it.
        sum_x2 = front_x**2 + rear_x**2
        self.roll_gain_front = front_share * (Mr * h) * front_x / sum_x2
        self.roll_gain_rear = rear_share * (Mr * h) * rear_x / sum_x2

        # -------------------------------------------------------------------
        # Mass matrix about the CoG: M = diag(Mr, Mr, Jzz)
        # -------------------------------------------------------------------
        self.M = np.diag([Mr, Mr, params['Jzz']])
        self.M_inv = np.linalg.inv(self.M)

        self.last_iterations = 0          # wheel solves the last step took (a report only)

    # =======================================================================
    # 2D cross product (a is 2xN and b is 2xN)
    # =======================================================================

    @staticmethod
    def cross2d(a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """
        z component of a x b, for 2D vectors a and b. Each is one vector [x, y], or an N x 2 array with
        one vector per row (x in column 0, y in column 1), which gives one result per row.

        From the determinant expansion:

            | i    j    k |
            | a_x  a_y  0 |
            | b_x  b_y  0 |

        = (a_x*b_y - a_y*b_x) k
        """
        return a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]

    # =======================================================================
    # Frame rotation
    # =======================================================================

    @staticmethod
    def _rotation(theta: float) -> np.ndarray:
        """
        R(theta), which rotates [x, y, yaw] from the local frame into the global frame.
        Its transpose rotates from the global frame back into the local frame.
        """
        return np.array([[np.cos(theta), -np.sin(theta), 0.0],
                         [np.sin(theta),  np.cos(theta), 0.0],
                         [0.0,            0.0,           1.0]])

    # =======================================================================
    # Forces
    # =======================================================================

    def compute_dynamic_normal_forces(self, state: OmniRobotStateODESolve) -> np.ndarray:
        """
        Calculates Step 5.2: the vertical load on each wheel, from the static split, the pitch and roll
        load transfer of the previous step's acceleration, and the structural vibration noise.
        [CHANGE 1, 6]

        Using the previous step's acceleration avoids an algebraic loop: the loads set the forces, which
        set the acceleration.
        """
        sigma_V = self.p['sigma_V']

        ax_past = state.a_local_past[0]   # right
        ay_past = state.a_local_past[1]   # forward

        # Pitch: accelerating forward moves load from the front wheels to the rear
        pitch_shift = self.pitch_gain * ay_past

        # Roll: accelerating right moves load from the right wheels to the left
        roll_shift_front = self.roll_gain_front * ax_past
        roll_shift_rear = self.roll_gain_rear * ax_past

        # Structural vibration on the loads
        w_F = self.rng.normal(0.0, sigma_V, 4) if sigma_V > 0 else np.zeros(4)

        F_n = np.array([
            self.F_n_front_static - pitch_shift - roll_shift_front + w_F[0],   # 1 FR
            self.F_n_front_static - pitch_shift + roll_shift_front + w_F[1],   # 2 FL
            self.F_n_rear_static + pitch_shift + roll_shift_rear + w_F[2],     # 3 RL
            self.F_n_rear_static + pitch_shift - roll_shift_rear + w_F[3],     # 4 RR
        ])

        # A wheel cannot pull on the floor
        return np.maximum(0.0, F_n)

    def compute_roller_friction(self, state: OmniRobotStateODESolve, F_n: np.ndarray) -> np.ndarray:
        """
        Calculates Step 5.8: the roller (a-axis) friction wrench on the chassis, from the chassis
        velocity of this step's start. [CHANGE 9]

        It lies along each wheel's axial direction, so it cannot turn the wheel. Explicit, one step
        late: that is stable because this force is not stiff (at most about 160 /s against 1/dt = 1000 /s).

        Returns [force x, force y, moment] on the chassis about the CoG [N, N, N m]; zero when
        roller_friction is off.
        """
        if not self.p['roller_friction']:
            return np.zeros(3)

        mu_a_s = self.p['mu_a_s']
        mu_a_k = self.p['mu_a_k']
        alpha_a = self.p['alpha_a']
        k_a = self.p['roller_smoothing']

        # Axial slip of each wheel: v_slip_a = H_axial v_local (the wheel's spin adds nothing along its axle)
        v_slip_a = self.H_axial @ state.v_local

        # Roller friction coefficient: mu_a_s at zero axial slip, tending to mu_a_k
        mu_a = mu_a_k + (mu_a_s - mu_a_k) * np.exp(-alpha_a * np.abs(v_slip_a))

        # (2/pi) atan(k v) is a smoothed sign(v)
        F_a = -F_n * mu_a * (2.0 / np.pi) * np.arctan(k_a * v_slip_a)

        # The wheels' forces and their moments about the CoG
        return self.H_axial.T @ F_a

    # =======================================================================
    # Wheel solve: drive friction (modified Dahl), motor and wheel together
    # =======================================================================
    #
    # Each wheel has a bristle deflection z, as a fraction of lugre_deflection (delta). At zero slip the
    # bristles hold their load, so the force does not drop to zero. In steady sliding the force is the
    # Stribeck curve mu_d F_n. v_s is the drive slip and v_c the creep speed (lugre_creep).
    #
    #     mu_d = mu_k + (mu_s - mu_k) exp(-alpha_d |v_s|)
    #     delta dz/dt = v_s - (mu_s / mu_d) sqrt(v_s^2 + v_c^2) z
    #     F = -mu_s F_n z
    #
    # The full LuGre model has other terms (bristle damping and viscous friction). They are removed here:
    # all this model needs is the bristle deflection term, so that the force is not zero at zero slip.
    # This makes it a modified Dahl model: LuGre with sigma_1 = sigma_2 = 0, as in Dahl, but keeping the
    # Stribeck curve, where strict Dahl sets mu_s = mu_d. Reference:
    # https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm
    #
    # The bristles and the wheel form a stiff loop (w*dt ~ 3.8), so they are solved implicitly with
    # backward Euler. [CHANGE 5]

    def compute_contact_friction_interface(self, state: OmniRobotStateODESolve, V: np.ndarray, F_n: np.ndarray) -> tuple:
        """
        Calculates Steps 5.3 to 5.7: the floor friction noise, the rolling drag, then each wheel's motor
        current, wheel speed and bristle deflection solved together. [CHANGE 5, 7]

        The friction noise (Dana's floor friction variation) is drawn once per step and held for every
        pass of the solve. The Stribeck coefficient and the bristle relax rate depend on the new slip,
        so the solve is repeated with them updated until the slip settles.

        V: applied motor voltages [V]
        F_n: from compute_dynamic_normal_forces [N]

        Returns (I_next, omega_next, z_next, F_floor, T_roll, solves):
            F_floor: force of the floor on each wheel, along its drive direction [N]
            T_roll: rolling drag on each wheel [N m]
            solves: how many times the solve ran
        """
        dt = self.dt
        r = self.p['r']
        Lm = self.p['Lm']
        R = self.p['R']
        Kb = self.p['Kb']
        Kt = self.p['Kt']
        Jw = self.p['Jw']
        Crr = self.p['Crr']
        roll_smoothing = self.p['roll_smoothing']
        mu_s_nom = self.p['mu_s']
        mu_k_nom = self.p['mu_k']
        sigma_mu = self.p['sigma_mu']
        alpha_d = self.p['alpha_d']
        delta = self.p['lugre_deflection']
        v_c = self.p['lugre_creep']
        tol = self.p['implicit_tol']
        max_iter = self.p['implicit_max_iter']

        # Floor friction variation for this step
        w_mu = self.rng.normal(0.0, sigma_mu, 4) if sigma_mu > 0 else np.zeros(4)
        mu_s = np.maximum(0.0, mu_s_nom + w_mu)
        mu_k = np.maximum(0.0, mu_k_nom + w_mu)

        # Wheel centre velocity along each drive direction, held over the step
        v_center_d = self.H_drive @ state.v_local

        # Step 5.3: rolling drag, smoothed through zero speed
        T_roll = Crr * F_n * r * np.tanh(state.omega / roll_smoothing)

        # Drive slip at the start of the step: v_s = H_drive v_local - r omega
        v_s = v_center_d - r * state.omega
        for solves in range(1, max_iter + 1):
            # Steps 5.4 to 5.6 at the latest slip: Stribeck curve, bristle limit and relax rate
            mu_d = mu_k + (mu_s - mu_k) * np.exp(-alpha_d * np.abs(v_s))
            F_static = mu_s * F_n
            bristle_decay = (mu_s / mu_d) * np.sqrt(v_s**2 + v_c**2)

            # Step 5.7: the motor, wheel and bristle equations of each wheel at the end of the step
            # (backward Euler), as the matrix equation A x = b with x = [I_next, omega_next, z_next]:
            #
            #     [ L/dt + R    Kb      0                        ] [ I_next     ]   [ L/dt I + V              ]
            #     [ -Kt         J/dt    -r F_static              ] [ omega_next ] = [ J/dt omega - T_roll     ]
            #     [ 0           r       delta/dt + bristle_decay ] [ z_next     ]   [ delta/dt z + v_center_d ]
            #
            # Solved by elimination, for all four wheels at once: the motor row gives I_next and the
            # bristle row gives z_next in terms of omega_next; putting both into the wheel row leaves one
            # equation for omega_next. Same answer as solving the 3x3 matrix.
            a_motor = Lm / dt + R                          # motor row, I_next coefficient
            a_bristle = delta / dt + bristle_decay         # bristle row, z_next coefficient
            b_motor = Lm / dt * state.I + V                # right-hand sides
            b_wheel = Jw / dt * state.omega - T_roll
            b_bristle = delta / dt * state.z + v_center_d

            omega_next = ((b_wheel + Kt * b_motor / a_motor + r * F_static * b_bristle / a_bristle)
                          / (Jw / dt + Kt * Kb / a_motor + r * r * F_static / a_bristle))
            I_next = (b_motor - Kb * omega_next) / a_motor
            z_next = (b_bristle - r * omega_next) / a_bristle

            # Stop once the new slip has stopped changing
            v_s_next = v_center_d - r * omega_next
            settled = np.max(np.abs(v_s_next - v_s)) < tol
            v_s = v_s_next
            if settled:
                break

        # Step 5.6: F = -mu_s F_n z
        F_floor = -F_static * z_next
        return I_next, omega_next, z_next, F_floor, T_roll, solves

    # =======================================================================
    # Chassis
    # =======================================================================

    def advance_chassis_kinetics(self, state: OmniRobotStateODESolve, F_floor: np.ndarray, wrench_roller: np.ndarray) -> OmniRobotStateODESolve:
        """
        Calculates Step 5.9: the chassis acceleration from Newton's law, then the global velocity and
        pose with forward Euler. [CHANGE 2]

        The wheels are fixed to the chassis, so their forces start in the local frame. They are rotated
        into the global frame, where Newton's law holds, and the global velocity is the advanced state.
        Advancing the local velocity instead leaves out the rotation of the local axes, and a robot that
        moves while it turns follows a wrong, curved path.

        Returns a new OmniMotorState with the chassis part filled in.
        """
        dt = self.dt
        theta = state.pos_global[2]
        R_theta = self._rotation(theta)

        # Sum of forces on the chassis, local frame, then rotated into the global frame
        F_local = self.H_drive.T @ F_floor + wrench_roller
        F_global = R_theta @ F_local

        # Newton's law about the CoG, global frame
        a_global = self.M_inv @ F_global

        # Forward Euler: velocity, then position
        v_global_next = state.v_global + dt * a_global
        pos_global_next = state.pos_global + dt * v_global_next

        next_state = OmniRobotStateODESolve()
        next_state.v_global = v_global_next
        next_state.pos_global = pos_global_next
        next_state.v_local = self._rotation(pos_global_next[2]).T @ v_global_next   # back into the local frame, new heading
        next_state.a_local_past = R_theta.T @ a_global                              # for the next step's load transfer
        return next_state

    # =======================================================================
    # One step, and what the controller sees
    # =======================================================================

    def apply_driver_limits(self, state: OmniRobotStateODESolve, V: np.ndarray) -> np.ndarray:
        """
        BLDC: the motor drivers' current limits, applied as a limit on the voltage. Each driver keeps its own motor
        current within I_max, and the four together keep the battery current within I_bus_max.

        The current settles within about L/R = 0.55 ms, less than one 1 ms step, so the current a voltage gives is
        taken as its steady value with this step's wheel speed:  i = (V - Kb w) / R
          1. Per motor:  |i| <= I_max, so V stays within Kb w +- R I_max.
          2. Battery:    I_bus = sum(V_i i_i) / V_bus <= I_bus_max. If over, every motor current is scaled by the
                         same s < 1 (V_i = Kb w_i + R s i_i), with s from
                             s sum(Kb w_i i_i) + s^2 R sum(i_i^2) = V_bus I_bus_max
        Returns the voltages the motors get.
        """
        R = self.p['R']
        back_emf = self.p['Kb'] * state.omega

        # 1. Per motor
        i = np.clip((V - back_emf) / R, -self.p['I_max'], self.p['I_max'])

        # 2. Battery
        P_limit = self.p['V_bus'] * self.p['I_bus_max']
        a = R * np.sum(i**2)
        b = np.sum(back_emf * i)
        if a + b > P_limit:
            s = (-b + np.sqrt(b**2 + 4.0 * a * P_limit)) / (2.0 * a)
            i = s * i

        V_applied = back_emf + R * i
        return V_applied

    def step(self, state: OmniRobotStateODESolve, V_cmd: np.ndarray) -> tuple:
        """
        Executes the whole discrete pipeline for one time step dt, with the voltages and the noise held
        over the step.

        state: OmniMotorState at step k
        V_cmd: motor voltage commands [V]

        Returns (next_state, T_load, S_status):
            next_state: OmniMotorState at step k+1
            T_load: load torque on each motor [N m]
            S_status: wheels whose slip this step is above slip_band (a report only)
        """
        V_max = self.p['V_max']
        r = self.p['r']
        

        # Step 5.1: supply limit [CHANGE 8]
        V = np.clip(np.asarray(V_cmd, dtype=float), -V_max, V_max)
        V = self.apply_driver_limits(state, V)      # BLDC: the drivers' per-motor and battery current limits

        F_n = self.compute_dynamic_normal_forces(state)
        wrench_roller = self.compute_roller_friction(state, F_n)
        I_next, omega_next, z_next, F_floor, T_roll, self.last_iterations = \
            self.compute_contact_friction_interface(state, V, F_n)

        next_state = self.advance_chassis_kinetics(state, F_floor, wrench_roller)
        next_state.I = I_next
        next_state.omega = omega_next
        next_state.z = z_next
        next_state.V_actual = V

        # Load torque on each motor
        T_load = F_floor * r + T_roll

        # Slip flags: drive slip v_s = H_drive v_local - r omega, with the new wheel speeds
        v_slip_d = self.H_drive @ state.v_local - r * omega_next
        S_status = np.abs(v_slip_d) > self.p['slip_band']      # CHANGED: was 0.05, ten times slip_band
        next_state.S = S_status

        return next_state

    def get_observable_state( self, full_state: OmniRobotStateODESolve) -> OmniRobotObservableState: 
        obs = OmniRobotObservableState() 
        obs.pos_global = full_state.pos_global.copy() 
        obs.omega = full_state.omega.copy() 
        obs.I = full_state.I.copy() 
        obs.V_actual = full_state.V_actual.copy()
        return obs

## Simulation Paramaters 
Any model paramater must be added to the paramater dictionary since this one dictionary gets passed to all plant models

In [ ]:
# 1. Define the simulation parameters dictionary (Henry calibrated constants)
simulation_parameters = {
    # Time Domain Constraints
    'dt': 0.001,               # Loop sampling interval (1 kHz loop rate)

    # Physical Constants
    'g': 9.81,                 # Gravitational acceleration (m/s^2)

    # System Power & Actuator Hardpoint Limits
    'V_max': 13.44,            # BLDC: largest applied voltage, PWM_LIMIT 0.8 x 16.8 V bus (Volts)
    'V_bus': 16.8,             # BLDC: battery bus voltage (Volts)
    'I_bus_max': 20.0,         # BLDC: battery current limit for the whole robot (Amperes)
    'I_max': 17.3,             # BLDC: per-motor driver limit, Simovr Iq 20 A as DC-equivalent current (sqrt(3)/2 x 20) (Amperes)

    # Robot Mechanical Properties
    'Mr': 2.698,               # Total mass of the robot chassis (kg)
    'Jzz': 0.007726,           # Chassis rotational inertia around Z-axis (kg*m^2)
    'Rb': 0.0795,              # Structural chassis base radius (meters)
    'h': 0.0611,               # Height of the Center of Mass above ground (meters)

    # Omni-Wheel Geometry
    'r': 0.0299,               # Operating radius of individual omni-wheels (meters)
    'Jw': 16.576e-6,        # BLDC: wheel and rotor inertia, direct drive (robot_constants WHEEL_INERTIA) (kg*m^2)
    'alpha_f': np.radians(30.0),# Front wheels mounting angle (converted to radians)
    'alpha_b': np.radians(45.0),# Rear wheels mounting angle (converted to radians)

    # DC Motor Electromechanical Properties
    'R': 0.24,                 # BLDC: line-to-line winding resistance, 5010 360 KV (Ohms)
    'Lm': 0.133e-3,             # BLDC: line-to-line winding inductance (Henries, 0.133 mH)
    'Kt': 60 / (2 * np.pi * 360),  # BLDC: torque constant 60 / (2 pi KV) = 0.0265 (Nm/A), direct drive
    'Kb': 60 / (2 * np.pi * 360),  # BLDC: back-EMF constant, equal to Kt (V/(rad/s))

    # Surface Contact and Friction Parameters
    'Crr': 0.025,              # Coefficient of rolling resistance
    'mu_s': 0.9,              # Nominal static coefficient of floor friction
    'mu_k': 0.4,             # Nominal kinetic coefficient of floor friction

    # Modified Dahl Floor Friction (LuGre bristle state with the Stribeck curve)
    'lugre_deflection': 2e-6,  # Presliding bristle deflection limit (m)
    'lugre_creep': 0.0001,     # Regularization velocity (m/s)
    'alpha_d': 1.0,            # Stribeck decay scale (s/m)
    'roll_smoothing': 0.5,     # Rolling resistance tanh smoothing speed (rad/s)
    'slip_band': 0.005,              # Slip speed above which a wheel is flagged as slipping (m/s)

    # Environmental Disturbance Variations (Stochastic Noise)
    'sigma_V': 0.15,           # Structural vibration standard deviation (Newtons)
    'sigma_mu': 0.03,           # Floor patch friction variance standard deviation

    # Roller (a-axis) Friction 
    'roller_friction': True,        # On/off switch. False = free rollers
    'mu_a_s': 0.1,                   # Roller friction coefficient at zero axle slip
    'mu_a_k': 0.3,                 # Roller friction coefficient at high axle slip
    'alpha_a': 1.0,                  # Decay from mu_a_s to mu_a_k with axle slip speed (s/m)
    'roller_smoothing': 200.0,       # Slope k_a of the smoothed sign, (2/pi) atan(k_a v_a) (s/m)

    # Wheel Solver, backward Euler 
    'implicit_tol': 1e-4,            # Repeat the wheel solve until the slip changes by less than this (m/s)
    'implicit_max_iter': 50,         # Most wheel solves per step

    #variables only used in solver plant
    'cog_y': 0.0,                    # Center of Mass forward of the base centre (m). Centred for now; CAD value 0.0102
    
}



## Testing Model Initialization

In [ ]:
# 2. Instantiate the physics engine plant
robot_plant = OmniRobotSimulationExplicit(dict(simulation_parameters, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)

# 3. Instantiate a fresh initial state container
initial_state = OmniRobotStateExplicit()

# 4. Generate the Comprehensive Plant Status Report
print("=======================================================================")
print("                OMNI-WHEEL PHYSICAL PLANT INITIALIZATION REPORT        ")
print("=======================================================================")
print(f" TIME INTERRUPT  | Sampling Rate: {1.0/robot_plant.dt:.0f} Hz | dt: {robot_plant.dt} s")
print("-----------------------------------------------------------------------")
print(" CHASSIS MECHANICAL PROPERTIES:")
print(f"  • Total Vehicle Mass (Mr):        {robot_plant.p['Mr']:.3f} kg")
print(f"  • Center of Mass Height (h):      {robot_plant.p['h']:.3f} m")
print(f"  • Rotational Inertia (Jzz):       {robot_plant.p['Jzz']:.5f} kg·m²")
print(f"  • Structural Base Radius (Rb):    {robot_plant.p['Rb']:.4f} m")
print(f"  • Front Mounting Angle (α_f):     {np.degrees(robot_plant.p['alpha_f']):.1f}° ")
print(f"  • Rear  Mounting Angle (α_b):     {np.degrees(robot_plant.p['alpha_b']):.1f}° ")
print("-----------------------------------------------------------------------")
print(" ELECTROMECHANICAL DRIVE & HARDPOINT BOUNDS:")
print(f"  • Supply Voltage Limit (V_max):   {robot_plant.p['V_max']:.1f} V      | Over-Current Limit (I_max): {robot_plant.p['I_max']:.1f} A")
print(f"  • Winding Resistance (R):         {robot_plant.p['R']:.3f} Ω      | Winding Inductance (Lm):    {robot_plant.p['Lm']:.6f} H")
print(f"  • Torque Scale Constant (Kt):     {robot_plant.p['Kt']:.4f} Nm/A  | Back-EMF Scale (Kb):        {robot_plant.p['Kb']:.4f} V/(rad/s)")
print(f"  • Assembly Rotor Inertia (Jw):    {robot_plant.p['Jw']:.7f} kg·m² | Operating Tire Radius (r):  {robot_plant.p['r']:.4f} m")
print("-----------------------------------------------------------------------")
print(" ENVIRONMENTAL BOUNDARY CONSTRAINTS & STOCHASTIC DISTURBANCES:")
print(f"  • Base Rolling Friction Drag (Crr):   {robot_plant.p['Crr']:.4f}  | Vibrational Force Noise (σ_V): {robot_plant.p['sigma_V']:.3f} N")
print(f"  • Nominal Sliding Friction (μ_k):     {robot_plant.p['mu_k']:.3f} | Nominal Static Friction (μ_s): {robot_plant.p['mu_s']:.3f}")
print("=======================================================================")


# 🧪 Kinematic & Dynamic Verification Testing

This section executes open-loop verification trials to validate the electromechanical motor re-coupling, asymmetric kinematics mapping ($\mathbf{H}$), and stick-slip friction boundaries under hardware-verified sign directions. 

### 📐 Structural Testing Grid Architecture
The execution pipeline performs a **multi-block comparative analysis** across two distinct physical frame designs and environmental settings:
1. **Block 1: Symmetric Frame Verification:** Configures the plant layout to an identical 45° alignment across all wheel lines ($\alpha_f = 45^\circ$, $\alpha_b = 45^\circ$).
2. **Block 2: Asymmetric Frame Verification:** Reverts the plant setup back to your custom asymmetric layout ($\alpha_f = 30^\circ$, $\alpha_b = 45^\circ$).
3. **Block 3: Asymmetric Frame With Boosted Voltage Verification:** Keeps the asymmetric frame but boosts the voltage to operate the voltage is a more ideal range but also cause slipping. The previous two verifications focus on the voltage direction and general kinematic so voltage is very low.   

### 🕹️ Verification Test Input Sequences
To isolate each independent degree of freedom, the simulation applies precise step voltage patterns across the four wheel motors:
*   **Test 1: Forward Motion ($\mathbf{V} = [V, -V, -V, V]^\top$):** Applies positive voltages to the right-side motors ($1, 4$) and negative voltages to the left-side motors ($2, 3$). This drives all wheels to push the ground backward symmetrically, canceling out lateral side forces and accelerating the robot straight ahead along its local $+Y_r$ axis.
*   **Test 2: Lateral Side-Strafing ($\mathbf{V} = [-V, -V, V, V]^\top$):** Applies negative voltages to the front motors ($1, 2$) and positive voltages to the rear motors ($3, 4$). This mirrors the longitudinal forces across the front-to-back split line, causing forward and backward vectors to cancel out. The remaining lateral components combine to push the chassis cleanly sideways to the right along its local $+X_r$ axis.
*   **Test 3: Pure Rotational Spinning ($\mathbf{V} = [V, V, V, V]^\top$):** Applies uniform positive voltages to all four wheels simultaneously. Because every omni-wheel is mounted tangentially to the chassis base circle, spinning them all in the same rotational direction eliminates all linear translation vectors and creates a pure angular torque, spinning the robot counter-clockwise ($+\theta$) around its Center of Mass.

### 🔄 Dual-Pass State Conditions
To clearly isolate the degradation caused by environmental disturbances, each vehicle geometry executes a sequential dual-pass simulation:
*   **Ideal Pass (Reference Baseline):** Strips all noise configurations ($\sigma_V = 0.0$, $\sigma_\mu = 0.0$). The results render as soft, transparent curves representing perfect theoretical trajectories.
*   **Perturbed Pass (Real-World Overlay):** Activates a repeatable, seeded random number generator stream (`np.random.default_rng`) to overlay high-frequency structural chassis vibrations ($w_F$) on the wheel loads and spatial friction coefficient patches ($w_\mu$).

### 📊 Logged Telemetry Variables
Each configuration runs the three directional maneuvers back-to-back over `t_end` seconds (set in the cell below, with the voltages) and generates a standalone 3x2 grid capturing:
*   **Applied Motor Terminal Voltages ($V_{1\dots4}$):** Verifies step-input directional polarities.
*   **Wheel Angular Velocities ($\omega_{1\dots4}$):** Monitors electromechanical back-EMF variations and independent rotor spinout integrations ($J_w$).
*   **Chassis Local Frame Speeds ($\dot{x}_r, \dot{y}_r, \dot{\theta}$):** Evaluates translational and angular momentum unbalances inside the mass matrix universe ($\mathbf{M}^{-1}\mathbf{H}^\top\mathbf{F}_k$).
*   **Global Position Tracking ($X-Y$ Trajectory Plane):** Visualizes dead-reckoning coordinate transformation drift paths directly overlaying ideal performance vs. noisy tracking realities.
*   **Drive and Axial Friction ($F/F_n$):** Each wheel's drive friction (the bristle model) and axial roller friction (the transverse friction) as a share of its load, on the same $\pm 1$ scale as the friction validation figure. Just outside that range, the drive plot marks each wheel's grip from the plant's slip flags: $+1$ rolling, $-1$ sliding (drive slip above `slip_band`).

**Plants:** `model_choice` (cell below) picks the plant. With `model_choice = 2` (the ODESolve plant) the third row is the drive and axial friction above. With `model_choice = 1` (Dana's explicit plant, which does not log its friction) the third row is the contact slip velocity and the sliding flags instead.


In [ ]:
#### CHOOOSING THE MODEL TO RUN and RUN PARAMATERS
model_choice = 2 # 1 for explicit, 2 for Linear System of Equation solver
low_voltage = 1.5
t_end = 1.5  
bost_voltage = 6.0
slip_limit=0.1

In [ ]:
"""
Friction logging for the tests
==============================

The plant, unchanged, with each wheel's friction saved every step as a share of its load, F/F_n:
    drive: the floor force along the wheel's drive direction (the bristle model)
    axial: the roller friction along the wheel's axle (the transverse friction)
"""

import numpy as np


class FrictionLoggedRobot(OmniRobotSimulationODESolve):
    """The plant with each wheel's load, drive friction and axial friction saved after every step."""

    def compute_roller_friction(self, state, F_n):
        wrench_roller = super().compute_roller_friction(state, F_n)

        # compute_roller_friction returns only the wrench on the chassis, so the force on each wheel is
        # worked out again here with the same equations, and checked against that wrench
        if self.p['roller_friction']:
            v_slip_a = self.H_axial @ state.v_local
            mu_a = self.p['mu_a_k'] + (self.p['mu_a_s'] - self.p['mu_a_k']) * np.exp(-self.p['alpha_a'] * np.abs(v_slip_a))
            F_axial = -F_n * mu_a * (2.0 / np.pi) * np.arctan(self.p['roller_smoothing'] * v_slip_a)
        else:
            F_axial = np.zeros(4)
        assert np.allclose(self.H_axial.T @ F_axial, wrench_roller), "axial forces do not match the plant's roller wrench"

        self.F_n = F_n
        self.F_axial = F_axial
        return wrench_roller

    def compute_contact_friction_interface(self, state, V, F_n):
        I_next, omega_next, z_next, F_floor, T_roll, solves = super().compute_contact_friction_interface(state, V, F_n)
        self.F_drive = F_floor
        return I_next, omega_next, z_next, F_floor, T_roll, solves

    def friction_shares(self):
        """Drive and axial friction / F_n of each wheel this step; NaN for a wheel off the floor."""
        F_n = np.where(self.F_n > 0, self.F_n, np.nan)
        return self.F_drive / F_n, self.F_axial / F_n


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

nominal_sigma_V = simulation_parameters['sigma_V']
nominal_sigma_mu = simulation_parameters['sigma_mu']

dt = simulation_parameters['dt']
steps = int(t_end / dt)
t_vec = np.linspace(0, t_end, steps)

# X-Y plot scale in robot widths (the wheel contact circle, 2 Rb): each axis shows at least 4 units,
# so a robot that barely moves looks like it barely moves; a longer path widens that axis.
# The unit is one robot width, or a quarter of one for the low-voltage tests.
robot_width = 2.0 * simulation_parameters['Rb']


def robot_width_limits(path, unit=robot_width):
    """Axis limits centred on the path: 4 units, or the path plus one unit if longer."""
    centre = (path.max() + path.min()) / 2.0
    units = max(4.0, np.ceil((path.max() - path.min()) / unit) + 1.0)
    return centre - units * unit / 2.0, centre + units * unit / 2.0


maneuvers = [
    {"name": "Test 1: Forward Motion", "V": np.array([low_voltage, -low_voltage, -low_voltage, low_voltage])},
    {"name": "Test 2: Lateral Side-Strafing", "V": np.array([-low_voltage, -low_voltage, low_voltage, low_voltage])},
    {"name": "Test 3: Pure Rotational Spinning", "V": np.array([low_voltage, low_voltage, low_voltage, low_voltage])}
]

# REMOVED redundancy: line style 'ls' is isolated out of the style modifiers
style_ideal = {'alpha': 0.4, 'lw': 2.5}  
style_noisy = {'alpha': 1.0, 'lw': 1.2}  

# CONFIGURATION CHANGE: FORCE SYMMETRIC 45-45 DESIGN
simulation_parameters['alpha_f'] = np.radians(45.0)
simulation_parameters['alpha_b'] = np.radians(45.0)

print("Executing Block 1: Symmetric 45-45 Wheel Angle Configurations...")

for m in maneuvers:
    runs_data = {}
    for condition in ["Ideal", "With Disturbances"]:
        if condition == "Ideal":
            simulation_parameters['sigma_V'] = 0.0
            simulation_parameters['sigma_mu'] = 0.0
        else:
            simulation_parameters['sigma_V'] = nominal_sigma_V
            simulation_parameters['sigma_mu'] = nominal_sigma_mu
            
        if model_choice == 1:
            robot_plant = OmniRobotSimulationExplicit(dict(simulation_parameters, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)
            state = OmniRobotStateExplicit()
        elif model_choice == 2:
            robot_plant = FrictionLoggedRobot(simulation_parameters, random_seed=42)   # the ODESolve plant, logging its friction
            state = OmniRobotStateODESolve()
        
        v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
        drive_hist, axial_hist, sliding_hist = [], [], []
        
        for k in range(steps):
            v_in_hist.append(m["V"].copy())
            omega_hist.append(state.omega.copy())
            v_local_hist.append(state.v_local.copy())
            pos_global_hist.append(state.pos_global.copy())
            state = robot_plant.step(state, m["V"])
            if model_choice == 2:
                drive_share, axial_share = robot_plant.friction_shares()
                drive_hist.append(drive_share); axial_hist.append(axial_share); sliding_hist.append(state.S.copy())
            
        runs_data[condition] = {
            "V": np.array(v_in_hist), "omega": np.array(omega_hist),
            "v_local": np.array(v_local_hist), "pos_global": np.array(pos_global_hist),
            "drive": np.array(drive_hist), "axial": np.array(axial_hist), "sliding": np.array(sliding_hist),
            "v_s": (np.array(v_local_hist) @ robot_plant.H.T) - robot_plant.p['r'] * np.array(omega_hist)
        }
        
    ideal = runs_data["Ideal"]
    noisy = runs_data["With Disturbances"]

    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    fig.suptitle(f"Symmetric Block (45-45 Alignment) | {m['name']}", fontsize=16, fontweight='bold')
    
    # Subplot 1: Voltage Inputs
    for w in range(4): 
        axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
    axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
    
    # Subplot 2: Wheel Speeds (FIXED: Raw string notation used)
    for w in range(4):
        axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
        axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
    axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 3: Local Frame Chassis Velocities (FIXED: Isolated style configuration)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
    axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 4: Global Trajectory Plane
    axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
    axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
    axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
    axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
    # Low voltage, small motion: X-Y scale in quarter robot widths
    path_x = np.concatenate((ideal["pos_global"][:, 0], noisy["pos_global"][:, 0]))
    path_y = np.concatenate((ideal["pos_global"][:, 1], noisy["pos_global"][:, 1]))
    axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_xlim(*robot_width_limits(path_x, robot_width / 4)); axs[1, 1].set_ylim(*robot_width_limits(path_y, robot_width / 4)); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
    
    if model_choice == 2:
        # Subplots 5 and 6: Friction as a share of each wheel's load, drive (bristle model) and axial (rollers, transverse)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
            axs[2, 1].plot(t_vec, ideal["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 1].plot(t_vec, noisy["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
        for ax, title in ((axs[2, 0], "Drive Friction (Bristle Model)"), (axs[2, 1], "Axial Friction (Rollers, Transverse)")):
            ax.set_title(title); ax.set_xlabel("Time (s)"); ax.set_ylabel(r"Friction / normal load, $F/F_n$"); ax.set_ylim(-1.0, 1.0); ax.grid(True); ax.legend(loc="upper right", ncol=2, fontsize='small')
    
        # Grip, from the plant's slip flags (drive slip above slip_band), drawn just outside the friction range so it
        # stays out of the way: +1 rolling (gripping), -1 sliding. Noisy run; wheels offset slightly so all four show.
        for w in range(4):
            level = 1.02 + 0.03 * w
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], np.nan, level), color=f'C{w}', lw=1.0)    # rolling
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], -level, np.nan), color=f'C{w}', lw=1.0)   # sliding
        axs[2, 0].set_ylim(-1.15, 1.15)
        axs[2, 0].text(0.01, 1.13, "rolling (+1)", transform=axs[2, 0].get_yaxis_transform(), va='top', fontsize='small')
        axs[2, 0].text(0.01, -1.13, "sliding (-1)", transform=axs[2, 0].get_yaxis_transform(), va='bottom', fontsize='small')
    else:
        # Dana's explicit plant: no friction logging, so the slip velocity and sliding flags
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

    plt.tight_layout()
    plt.show()

In [ ]:
# CONFIGURATION CHANGE: REVERT TO ASYMMETRIC 30-45 DESIGN
simulation_parameters['alpha_f'] = np.radians(30.0)
simulation_parameters['alpha_b'] = np.radians(45.0)

print("Executing Block 2: Asymmetric 30-45 Wheel Angle Configurations...")

for m in maneuvers:
    runs_data = {}
    for condition in ["Ideal", "With Disturbances"]:
        if condition == "Ideal":
            simulation_parameters['sigma_V'] = 0.0
            simulation_parameters['sigma_mu'] = 0.0
        else:
            simulation_parameters['sigma_V'] = nominal_sigma_V
            simulation_parameters['sigma_mu'] = nominal_sigma_mu
            
        if model_choice == 1:
            robot_plant = OmniRobotSimulationExplicit(dict(simulation_parameters, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)
            state = OmniRobotStateExplicit()
        elif model_choice == 2:
            robot_plant = FrictionLoggedRobot(simulation_parameters, random_seed=42)   # the ODESolve plant, logging its friction
            state = OmniRobotStateODESolve()
        
        v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
        drive_hist, axial_hist, sliding_hist = [], [], []
        
        for k in range(steps):
            v_in_hist.append(m["V"].copy())
            omega_hist.append(state.omega.copy())
            v_local_hist.append(state.v_local.copy())
            pos_global_hist.append(state.pos_global.copy())
            state = robot_plant.step(state, m["V"])
            if model_choice == 2:
                drive_share, axial_share = robot_plant.friction_shares()
                drive_hist.append(drive_share); axial_hist.append(axial_share); sliding_hist.append(state.S.copy())
            
        runs_data[condition] = {
            "V": np.array(v_in_hist), "omega": np.array(omega_hist),
            "v_local": np.array(v_local_hist), "pos_global": np.array(pos_global_hist),
            "drive": np.array(drive_hist), "axial": np.array(axial_hist), "sliding": np.array(sliding_hist),
            "v_s": (np.array(v_local_hist) @ robot_plant.H.T) - robot_plant.p['r'] * np.array(omega_hist)
        }
        
    ideal = runs_data["Ideal"]
    noisy = runs_data["With Disturbances"]

    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    fig.suptitle(f"Asymmetric Block (30-45 Alignment) | {m['name']}", fontsize=16, fontweight='bold')
    
    # Subplot 1: Voltage Inputs
    for w in range(4): 
        axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
    axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
    
    # Subplot 2: Wheel Speeds (FIXED: Raw string notation used)
    for w in range(4):
        axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
        axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
    axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 3: Local Frame Chassis Velocities (FIXED: Line parameters isolated)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
    axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 4: Global Trajectory Plane
    axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
    axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
    axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
    axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
    # Low voltage, small motion: X-Y scale in quarter robot widths
    path_x = np.concatenate((ideal["pos_global"][:, 0], noisy["pos_global"][:, 0]))
    path_y = np.concatenate((ideal["pos_global"][:, 1], noisy["pos_global"][:, 1]))
    axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_xlim(*robot_width_limits(path_x, robot_width / 4)); axs[1, 1].set_ylim(*robot_width_limits(path_y, robot_width / 4)); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
    
    if model_choice == 2:
        # Subplots 5 and 6: Friction as a share of each wheel's load, drive (bristle model) and axial (rollers, transverse)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
            axs[2, 1].plot(t_vec, ideal["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 1].plot(t_vec, noisy["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
        for ax, title in ((axs[2, 0], "Drive Friction (Bristle Model)"), (axs[2, 1], "Axial Friction (Rollers, Transverse)")):
            ax.set_title(title); ax.set_xlabel("Time (s)"); ax.set_ylabel(r"Friction / normal load, $F/F_n$"); ax.set_ylim(-1.0, 1.0); ax.grid(True); ax.legend(loc="upper right", ncol=2, fontsize='small')
    
        # Grip, from the plant's slip flags (drive slip above slip_band), drawn just outside the friction range so it
        # stays out of the way: +1 rolling (gripping), -1 sliding. Noisy run; wheels offset slightly so all four show.
        for w in range(4):
            level = 1.02 + 0.03 * w
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], np.nan, level), color=f'C{w}', lw=1.0)    # rolling
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], -level, np.nan), color=f'C{w}', lw=1.0)   # sliding
        axs[2, 0].set_ylim(-1.15, 1.15)
        axs[2, 0].text(0.01, 1.13, "rolling (+1)", transform=axs[2, 0].get_yaxis_transform(), va='top', fontsize='small')
        axs[2, 0].text(0.01, -1.13, "sliding (-1)", transform=axs[2, 0].get_yaxis_transform(), va='bottom', fontsize='small')
    else:
        # Dana's explicit plant: no friction logging, so the slip velocity and sliding flags
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

    plt.tight_layout()
    plt.show()

# Safely restore baseline parameter noise variables back to configuration dictionary
simulation_parameters['sigma_V'] = nominal_sigma_V
simulation_parameters['sigma_mu'] = nominal_sigma_mu

In [ ]:
# =======================================================================
# CONFIGURATION CHANGE: KEEP ASYMMETRIC 30-45 DESIGN BUT BOOSTED VOLTAGE
# =======================================================================
simulation_parameters['alpha_f'] = np.radians(30.0)
simulation_parameters['alpha_b'] = np.radians(45.0)


# Define high-voltage directional maneuvers matching your verified sign choices
high_volt_maneuvers = [
    {"name": "Test 1: Forward Motion (High Volt)", "V": np.array([bost_voltage, -bost_voltage, -bost_voltage, bost_voltage])},
    {"name": "Test 2: Lateral Side-Strafing (High Volt)", "V": np.array([-bost_voltage, -bost_voltage, bost_voltage, bost_voltage])},
    {"name": "Test 3: Pure Rotational Spinning (High Volt)", "V": np.array([bost_voltage, bost_voltage, bost_voltage, bost_voltage])}
]

print(f"Executing Block 3: Asymmetric 30-45 Wheel Angle Configurations at High Voltage ({bost_voltage:.1f}V)...")

for m in high_volt_maneuvers:
    runs_data = {}
    for condition in ["Ideal", "With Disturbances"]:
        if condition == "Ideal":
            simulation_parameters['sigma_V'] = 0.0
            simulation_parameters['sigma_mu'] = 0.0
        else:
            simulation_parameters['sigma_V'] = nominal_sigma_V
            simulation_parameters['sigma_mu'] = nominal_sigma_mu
            
        # Instantiate plant under custom asymmetric high-volt conditions (Seeded for reproducibility)
        if model_choice == 1:
            robot_plant = OmniRobotSimulationExplicit(dict(simulation_parameters, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)
            state = OmniRobotStateExplicit()
        elif model_choice == 2:
            robot_plant = FrictionLoggedRobot(simulation_parameters, random_seed=42)   # the ODESolve plant, logging its friction
            state = OmniRobotStateODESolve()
        
        v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
        drive_hist, axial_hist, sliding_hist = [], [], []
        
        for k in range(steps):
            v_in_hist.append(m["V"].copy())
            omega_hist.append(state.omega.copy())
            v_local_hist.append(state.v_local.copy())
            pos_global_hist.append(state.pos_global.copy())
            state = robot_plant.step(state, m["V"])
            if model_choice == 2:
                drive_share, axial_share = robot_plant.friction_shares()
                drive_hist.append(drive_share); axial_hist.append(axial_share); sliding_hist.append(state.S.copy())
            
        runs_data[condition] = {
            "V": np.array(v_in_hist), "omega": np.array(omega_hist),
            "v_local": np.array(v_local_hist), "pos_global": np.array(pos_global_hist),
            "drive": np.array(drive_hist), "axial": np.array(axial_hist), "sliding": np.array(sliding_hist),
            "v_s": (np.array(v_local_hist) @ robot_plant.H.T) - robot_plant.p['r'] * np.array(omega_hist)
        }
        
    ideal = runs_data["Ideal"]
    noisy = runs_data["With Disturbances"]

    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    fig.suptitle(f"Asymmetric High-Volt Block (30-45 @ 5.0V) | {m['name']}", fontsize=16, fontweight='bold')
    
    # Subplot 1: Voltage Inputs
    for w in range(4): 
        axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
    axs[0, 0].set_title("Applied Motor Terminal Voltages"); axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
    
    # Subplot 2: Wheel Speeds
    for w in range(4):
        axs[0, 1].plot(t_vec, ideal["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
        axs[0, 1].plot(t_vec, noisy["omega"][:, w], '-', color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
    axs[0, 1].set_title("Wheel Angular Velocities Comparison"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 3: Local Frame Chassis Velocities
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
    axs[1, 0].set_title("Chassis Local Frame Speeds"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 4: Global Trajectory Plane
    axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
    axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
    axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
    axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
    path_x = np.concatenate((ideal["pos_global"][:, 0], noisy["pos_global"][:, 0]))
    path_y = np.concatenate((ideal["pos_global"][:, 1], noisy["pos_global"][:, 1]))
    axs[1, 1].set_title("Global Spatial Position Tracking (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_xlim(*robot_width_limits(path_x)); axs[1, 1].set_ylim(*robot_width_limits(path_y)); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
    
    if model_choice == 2:
        # Subplots 5 and 6: Friction as a share of each wheel's load, drive (bristle model) and axial (rollers, transverse)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
            axs[2, 1].plot(t_vec, ideal["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 1].plot(t_vec, noisy["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
        for ax, title in ((axs[2, 0], "Drive Friction (Bristle Model)"), (axs[2, 1], "Axial Friction (Rollers, Transverse)")):
            ax.set_title(title); ax.set_xlabel("Time (s)"); ax.set_ylabel(r"Friction / normal load, $F/F_n$"); ax.set_ylim(-1.0, 1.0); ax.grid(True); ax.legend(loc="upper right", ncol=2, fontsize='small')
    
        # Grip, from the plant's slip flags (drive slip above slip_band), drawn just outside the friction range so it
        # stays out of the way: +1 rolling (gripping), -1 sliding. Noisy run; wheels offset slightly so all four show.
        for w in range(4):
            level = 1.02 + 0.03 * w
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], np.nan, level), color=f'C{w}', lw=1.0)    # rolling
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], -level, np.nan), color=f'C{w}', lw=1.0)   # sliding
        axs[2, 0].set_ylim(-1.15, 1.15)
        axs[2, 0].text(0.01, 1.13, "rolling (+1)", transform=axs[2, 0].get_yaxis_transform(), va='top', fontsize='small')
        axs[2, 0].text(0.01, -1.13, "sliding (-1)", transform=axs[2, 0].get_yaxis_transform(), va='bottom', fontsize='small')
    else:
        # Dana's explicit plant: no friction logging, so the slip velocity and sliding flags
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

    plt.tight_layout()
    plt.show()

# Final safe restoration of baseline parameter noise variables back to dictionary
simulation_parameters['sigma_V'] = nominal_sigma_V
simulation_parameters['sigma_mu'] = nominal_sigma_mu

# 🧪 Multi-Stage Sequence Verification Testing

This section executes verification routines using a **Multi-Stage Step Profile Timeline**. Each directional maneuver commands a voltage profile divided into three 1.0-second stages (Positive Drive $\to$ Negative Plug-Braking $\to$ Zero Coasting). This explicitly evaluates the plant's transient response, deceleration dynamics, and LuGre continuous friction behavior without artificial velocity snapping or numerical chatter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Save your configured noise parameters to toggle the passes cleanly
nominal_sigma_V = simulation_parameters['sigma_V']
nominal_sigma_mu = simulation_parameters['sigma_mu']

# Define a 3.0 second timeline split into three distinct 1.0s intervals
t_stage = 1.0
dt = simulation_parameters['dt']
steps_per_stage = int(t_stage / dt)
total_steps = steps_per_stage * 3
t_vec = np.linspace(0, t_stage * 3, total_steps)

# Helper function to dynamically generate the voltage vector at any given step index k
def get_profile_voltage(test_type, step_idx, steps_per_stage):
    # Determine which chronological stage we are currently executing
    if step_idx < steps_per_stage:
        stage = "Positive"
        mag = 4.0
    elif step_idx < 2 * steps_per_stage:
        stage = "Negative"
        mag = -4.0
    else:
        stage = "Zero"
        mag = 0.0
        
    # Apply your verified hardware directional sign combinations scaled by stage magnitude
    if test_type == "Forward/Backward":
        return np.array([mag, -mag, -mag, mag])
    elif test_type == "Side-to-Side":
        return np.array([mag, mag, -mag, -mag])
    elif test_type == "CCW/CW Rotation":
        return np.array([mag, mag, mag, mag])
    return np.zeros(4)

# Define our three complex multi-stage test scenarios
test_scenarios = ["Forward/Backward", "Side-to-Side", "CCW/CW Rotation"]

# Plotting configuration styles
style_ideal = {'ls': '-', 'alpha': 0.4, 'lw': 2.5}  
style_noisy = {'ls': '-', 'alpha': 1.0, 'lw': 1.2}  

# =======================================================================
# STRUCTURAL RUN 1: ASYMMETRIC 30-45 CONFIGURATION WITH PROFILE MANEUVERS
# =======================================================================
simulation_parameters['alpha_f'] = np.radians(30.0)
simulation_parameters['alpha_b'] = np.radians(45.0)

print("Executing Profile Tests under Asymmetric 30-45 Geometry...")

for test_name in test_scenarios:
    runs_data = {}
    
    for condition in ["Ideal", "With Disturbances"]:
        if condition == "Ideal":
            simulation_parameters['sigma_V'] = 0.0
            simulation_parameters['sigma_mu'] = 0.0
        else:
            simulation_parameters['sigma_V'] = nominal_sigma_V
            simulation_parameters['sigma_mu'] = nominal_sigma_mu
            
        if model_choice == 1:
            robot_plant = OmniRobotSimulationExplicit(dict(simulation_parameters, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)
            state = OmniRobotStateExplicit()
        elif model_choice == 2:
            robot_plant = FrictionLoggedRobot(simulation_parameters, random_seed=42)   # the ODESolve plant, logging its friction
            state = OmniRobotStateODESolve()
        
        v_in_hist, omega_hist, v_local_hist, pos_global_hist = [], [], [], []
        drive_hist, axial_hist, sliding_hist = [], [], []
        
        # Step through the full 3.0 second timeline
        for k in range(total_steps):
            V_cmd = get_profile_voltage(test_name, k, steps_per_stage)
            
            v_in_hist.append(V_cmd.copy())
            omega_hist.append(state.omega.copy())
            v_local_hist.append(state.v_local.copy())
            pos_global_hist.append(state.pos_global.copy())
            
            state = robot_plant.step(state, V_cmd)
            if model_choice == 2:
                drive_share, axial_share = robot_plant.friction_shares()
                drive_hist.append(drive_share); axial_hist.append(axial_share); sliding_hist.append(state.S.copy())
            
        runs_data[condition] = {
            "V": np.array(v_in_hist), "omega": np.array(omega_hist),
            "v_local": np.array(v_local_hist), "pos_global": np.array(pos_global_hist),
            "drive": np.array(drive_hist), "axial": np.array(axial_hist), "sliding": np.array(sliding_hist),
            "v_s": (np.array(v_local_hist) @ robot_plant.H.T) - robot_plant.p['r'] * np.array(omega_hist)
        }
        
    ideal = runs_data["Ideal"]
    noisy = runs_data["With Disturbances"]

    # -------------------------------------------------------------------
    # GENERATE 2x2 GRAPHS FOR COMPLEX PROFILE COUPLING
    # -------------------------------------------------------------------
    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    fig.suptitle(f"Asymmetric Chassis (30-45) | {test_name}: Pos/Neg/Zero Sequence", fontsize=16, fontweight='bold')
    
    # Subplot 1: Voltage Profiles (Now showing step changes over time)
    for w in range(4): 
        axs[0, 0].plot(t_vec, ideal["V"][:, w], label=f'V{w+1}')
    axs[0, 0].set_title("Applied Motor Voltage Timeline")
    axs[0, 0].set_xlabel("Time (s)"); axs[0, 0].set_ylabel("Voltage (V)"); axs[0, 0].grid(True); axs[0, 0].legend(loc="upper right")
    
    # Subplot 2: Wheel Speeds showing spikes during aggressive direction switches
    for w in range(4):
        axs[0, 1].plot(t_vec, ideal["omega"][:, w], color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Ideal', **style_ideal)
        axs[0, 1].plot(t_vec, noisy["omega"][:, w], color=f'C{w}', label=r'$\omega_{' + str(w+1) + r'}$ Noise', **style_noisy)
    axs[0, 1].set_title("Wheel Angular Velocities (LuGre Continuous Response)"); axs[0, 1].set_xlabel("Time (s)"); axs[0, 1].set_ylabel("Speed (rad/s)"); axs[0, 1].grid(True); axs[0, 1].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 3: Local Frame Velocities showing Braking/Deceleration Curves
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 0], 'r-', label=r'$\dot{x}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 1], 'b-', label=r'$\dot{y}_r$ Noise', **style_noisy)
    axs[1, 0].plot(t_vec, ideal["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Ideal', **style_ideal)
    axs[1, 0].plot(t_vec, noisy["v_local"][:, 2], 'g-', label=r'$\dot{\theta}$ Noise', **style_noisy)
    axs[1, 0].set_title("Chassis Local Frame Velocity Tracking"); axs[1, 0].set_xlabel("Time (s)"); axs[1, 0].set_ylabel("Velocity (m/s or rad/s)"); axs[1, 0].grid(True); axs[1, 0].legend(loc="upper right", ncol=2, fontsize='small')
    
    # Subplot 4: Global Trajectory Plane showing bidirectional path trace
    axs[1, 1].plot(ideal["pos_global"][:, 0], ideal["pos_global"][:, 1], 'k--', lw=2.0, label='Ideal Base Path')
    axs[1, 1].plot(noisy["pos_global"][:, 0], noisy["pos_global"][:, 1], 'b-', lw=1.5, label='Perturbed Path')
    axs[1, 1].plot(noisy["pos_global"][0, 0], noisy["pos_global"][0, 1], 'go', markersize=8, label='Start')
    axs[1, 1].plot(noisy["pos_global"][steps_per_stage, 0], noisy["pos_global"][steps_per_stage, 1], 'y^', markersize=8, label='Switch to Neg')
    axs[1, 1].plot(noisy["pos_global"][2*steps_per_stage, 0], noisy["pos_global"][2*steps_per_stage, 1], 'mo', markersize=8, label='Switch to Zero')
    axs[1, 1].plot(noisy["pos_global"][-1, 0], noisy["pos_global"][-1, 1], 'rx', markersize=8, label='End')
    path_x = np.concatenate((ideal["pos_global"][:, 0], noisy["pos_global"][:, 0]))
    path_y = np.concatenate((ideal["pos_global"][:, 1], noisy["pos_global"][:, 1]))
    axs[1, 1].set_title("Global Space Position Motion (X-Y)"); axs[1, 1].set_xlabel("Global X Position (m)"); axs[1, 1].set_ylabel("Global Y Position (m)"); axs[1, 1].set_xlim(*robot_width_limits(path_x)); axs[1, 1].set_ylim(*robot_width_limits(path_y)); axs[1, 1].grid(True); axs[1, 1].legend(loc="best")
    
    if model_choice == 2:
        # Subplots 5 and 6: Friction as a share of each wheel's load, drive (bristle model) and axial (rollers, transverse)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["drive"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
            axs[2, 1].plot(t_vec, ideal["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Ideal', **style_ideal)
            axs[2, 1].plot(t_vec, noisy["axial"][:, w], color=f'C{w}', label=f'Wheel {w+1} Noise', **style_noisy)
        for ax, title in ((axs[2, 0], "Drive Friction (Bristle Model)"), (axs[2, 1], "Axial Friction (Rollers, Transverse)")):
            ax.set_title(title); ax.set_xlabel("Time (s)"); ax.set_ylabel(r"Friction / normal load, $F/F_n$"); ax.set_ylim(-1.0, 1.0); ax.grid(True); ax.legend(loc="upper right", ncol=2, fontsize='small')
    
        # Grip, from the plant's slip flags (drive slip above slip_band), drawn just outside the friction range so it
        # stays out of the way: +1 rolling (gripping), -1 sliding. Noisy run; wheels offset slightly so all four show.
        for w in range(4):
            level = 1.02 + 0.03 * w
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], np.nan, level), color=f'C{w}', lw=1.0)    # rolling
            axs[2, 0].plot(t_vec, np.where(noisy["sliding"][:, w], -level, np.nan), color=f'C{w}', lw=1.0)   # sliding
        axs[2, 0].set_ylim(-1.15, 1.15)
        axs[2, 0].text(0.01, 1.13, "rolling (+1)", transform=axs[2, 0].get_yaxis_transform(), va='top', fontsize='small')
        axs[2, 0].text(0.01, -1.13, "sliding (-1)", transform=axs[2, 0].get_yaxis_transform(), va='bottom', fontsize='small')
    else:
        # Dana's explicit plant: no friction logging, so the slip velocity and sliding flags
        # Subplot 5: Wheel Contact Slip Velocity (Option 2)
        for w in range(4):
            axs[2, 0].plot(t_vec, ideal["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Ideal', **style_ideal)
            axs[2, 0].plot(t_vec, noisy["v_s"][:, w], color=f'C{w}', label=f'$v_{{s,{w+1}}}$ Noise', **style_noisy)
        axs[2, 0].axhline(slip_limit, color='r', linestyle='--', alpha=0.7, label=f'Sliding Limit (±{slip_limit*1000:.1f} mm/s)')
        axs[2, 0].axhline(-slip_limit, color='r', linestyle='--', alpha=0.7)
        axs[2, 0].set_title("Wheel Contact Slip Velocity (v_s in m/s)")
        axs[2, 0].set_xlabel("Time (s)"); axs[2, 0].set_ylabel("Slip Velocity (m/s)")
        axs[2, 0].grid(True); axs[2, 0].legend(loc="upper right", ncol=2, fontsize='small')

        # Subplot 6: Wheel Active Sliding Binary Indicator (|v_s| > 5 mm/s)
        for w in range(4):
            axs[2, 1].plot(t_vec, (np.abs(ideal["v_s"][:, w]) > slip_limit).astype(float), color=f'C{w}', label=f'Wheel {w+1} Sliding')
        axs[2, 1].set_title(f"Wheel Active Sliding Status Indicator (|v_s| > {slip_limit*1000:.1f} mm/s)")
        axs[2, 1].set_xlabel("Time (s)"); axs[2, 1].set_ylabel("Sliding Flag (1=Sliding, 0=Rolling)")
        axs[2, 1].set_ylim(-0.1, 1.1)
        axs[2, 1].grid(True); axs[2, 1].legend(loc="upper right", ncol=2, fontsize='small')

    plt.tight_layout()
    plt.show()

# Restore baseline parameters securely
simulation_parameters['sigma_V'] = nominal_sigma_V
simulation_parameters['sigma_mu'] = nominal_sigma_mu

## Friction validation

Our plant's drive friction against Coulomb friction from slip speed, with the Stribeck curves $\pm\mu_d(v_s)$ overlaid.

- **Coulomb:** $F/F_n = \mu_d(v_s)\,\frac{2}{\pi}\arctan(k\,v_s)$, friction as a function of slip speed only. The arctangent is a smoothed sign, with the same slope $k$ as the roller friction.
- **Our plant:** the modified Dahl bristle model. Each wheel's drive slip and floor force are logged every step while the robot runs full-voltage reversals, so the wheels slide in both directions.


In [ ]:
"""
Friction validation
===================

Runs the plant unchanged and logs each wheel's drive slip and friction every step, then plots them
against Coulomb friction from slip speed and the Stribeck curves +/- mu_d(v_s).
"""

import numpy as np
import matplotlib.pyplot as plt

friction_parameters = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0)   # a copy: the tests above change simulation_parameters   # no noise: the friction law alone
mu_s = friction_parameters['mu_s']
mu_k = friction_parameters['mu_k']
alpha_d = friction_parameters['alpha_d']
k_atan = friction_parameters['roller_smoothing']    # arctan slope, the same as the roller friction (s/m)


def mu_d(v_s):
    """Stribeck curve: mu_s at zero slip, tending to mu_k."""
    return mu_k + (mu_s - mu_k) * np.exp(-alpha_d * np.abs(v_s))


def coulomb_friction(v_s):
    """Coulomb friction from slip speed, / F_n: mu_d(v_s) times the smoothed sign (2/pi) atan(k v_s)."""
    return mu_d(v_s) * (2.0 / np.pi) * np.arctan(k_atan * v_s)


class LoggedRobot(OmniRobotSimulationODESolve):
    """The plant, unchanged, with each wheel's load, floor force and drive slip saved after every wheel solve."""
    def compute_contact_friction_interface(self, state, V, F_n):
        I_next, omega_next, z_next, F_floor, T_roll, solves = super().compute_contact_friction_interface(state, V, F_n)
        self.F_n = F_n
        self.F_floor = F_floor
        self.v_s = self.H_drive @ state.v_local - self.p['r'] * omega_next   # same slip as the slip flags
        return I_next, omega_next, z_next, F_floor, T_roll, solves


# -------------------------------------------------------------------
# Run: full voltage, flipped every 0.5 s, so the wheels slide both ways
# -------------------------------------------------------------------
friction_robot = LoggedRobot(friction_parameters)
friction_state = OmniRobotStateODESolve()
V_slide = np.array([12.0, -12.0, -12.0, 12.0])

slip = []
friction = []
for k in range(3000):
    V = V_slide if (k // 500) % 2 == 0 else -V_slide
    friction_state = friction_robot.step(friction_state, V)

    loaded = friction_robot.F_n > 0                  # a lifted wheel has no friction coefficient
    slip.append(friction_robot.v_s[loaded])
    friction.append(-friction_robot.F_floor[loaded] / friction_robot.F_n[loaded])   # against the slip, / F_n

slip = np.concatenate(slip)
friction = np.concatenate(friction)

# -------------------------------------------------------------------
# Plot: the full slip range, and a zoom near zero slip
# -------------------------------------------------------------------
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(10, 4.8), gridspec_kw={'width_ratios': [1.35, 1]})

for ax, v_max, every in ((ax_full, 3.0, 7), (ax_zoom, 0.03, 1)):
    v = np.linspace(-v_max, v_max, 4001)
    near = np.abs(slip) <= v_max

    ax.plot(v, mu_d(v), color='#52514e', lw=1.0, ls='--', label=r'$\pm\mu_d(v_s)$')
    ax.plot(v, -mu_d(v), color='#52514e', lw=1.0, ls='--')
    ax.plot(v, coulomb_friction(v), color='#eb6834', lw=1.8, label=r'Coulomb: $-F/F_n = \mu_d(v_s)\,\frac{2}{\pi}\arctan(k v_s)$, a function of the slip')
    ax.scatter(slip[near][::every], friction[near][::every], s=8, color='#2a78d6', alpha=0.5, lw=0, label=r'Our plant, bristle model: $-F/F_n = \mu_s z$, $z$ a state (the slip is a result)')

    ax.axhline(0.0, color='#52514e', lw=0.6)
    ax.axvline(0.0, color='#52514e', lw=0.6)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xlabel(r'Drive slip $v_s$ (m/s)')
    ax.grid(True, color='#e4e3df')

ax_full.set_ylabel(r'Friction / normal load, $-F/F_n$')
ax_full.set_title('(a) Full slip range')
ax_zoom.set_title(r'(b) Near zero slip, $|v_s| < 30$ mm/s')

handles, labels = ax_full.get_legend_handles_labels()
legend = fig.legend(handles, labels, loc='lower center', ncol=1, frameon=False)
for handle in legend.legend_handles:
    handle.set_alpha(1.0)

fig.tight_layout(rect=(0, 0.16, 1, 1))
plt.show()
